[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sunilmogadati/systems-in-production/blob/main/implementation/notebooks/AI_Engineer_Accelerator_Agents.ipynb) &nbsp; [![View on GitHub](https://img.shields.io/badge/GitHub-View%20Source-blue?logo=github)](https://github.com/sunilmogadati/systems-in-production/blob/main/implementation/notebooks/AI_Engineer_Accelerator_Agents.ipynb)

# AI (Artificial Intelligence) Agents — Teaching AI to Think, Plan, and Act

**Program:** AI Engineer Accelerator | **Week 15: AI Agents**
**Author:** Sunil Mogadati

---

Agents are the bridge from retrieval to action. RAG (Retrieval-Augmented Generation) finds information. Multimodal AI perceives the world in different formats. **Agents decide what to do with it.** This notebook teaches you to build AI that reasons, plans, uses tools, and acts — then observes the result and decides what to do next.

**What you will build:**
1. **A ReAct agent from scratch** — the industry-standard Thought → Action → Observation loop
2. **A Tree of Thought planner** — explore multiple reasoning paths before committing
3. **A multi-agent router system** — specialized agents that collaborate on complex tasks
4. **An MCP server and client** — the emerging standard for agent-tool communication
5. **A production-ready AgentPipeline class** — logging, retries, cost tracking, evaluation

**Time:** ~90–120 minutes

**Prerequisites:**
- [Build a RAG System from Scratch](https://colab.research.google.com/github/sunilmogadati/systems-in-production/blob/main/implementation/notebooks/AI_Engineer_Accelerator_RAG_from_Scratch.ipynb) — you should understand embeddings, vector databases, and retrieval
- [Beyond Text: Multimodal AI](https://colab.research.google.com/github/sunilmogadati/systems-in-production/blob/main/implementation/notebooks/AI_Engineer_Accelerator_Multimodal_AI.ipynb) — you should understand multi-modal pipelines and tool-like patterns

**Models used (all local, no API (Application Programming Interface) keys):**
- `mistral` via Ollama — reasoning, planning, tool selection

## Key Terms — Abbreviations Used in This Notebook

Every abbreviation below is explained when first used, but this is your quick reference:

| Abbreviation | Full Form |
|:------------|:----------|
| **AI** | Artificial Intelligence |
| **API** | Application Programming Interface |
| **AWS** | Amazon Web Services |
| **CoT** | Chain of Thought |
| **GCP** | Google Cloud Platform |
| **HTTP** | HyperText Transfer Protocol |
| **JSON** | JavaScript Object Notation |
| **LLMs** | Large Language Models |
| **LLM** | Large Language Model |
| **MCP** | Model Context Protocol |
| **PTO** | Paid Time Off |
| **RAG** | Retrieval-Augmented Generation |
| **REST** | Representational State Transfer |
| **SQL** | Structured Query Language |
| **SSE** | Server-Sent Events |
| **ToT** | Tree of Thought |
| **PII** | Personally Identifiable Information |


## Table of Contents

### Part I: What Is an Agent? (S0–S5)
0. [You Already Use AI Agents Every Day](#0-you-already-use-ai-agents-every-day) — entry frame & chef analogy
1. [The Agent Loop](#1-the-agent-loop) — Thought → Action → Observation → Repeat
2. [Hello World Agent](#2-hello-world-agent) — your first agent in ~30 lines
3. ["Why Does This Work?" — LLMs (Large Language Models) as Reasoning Engines](#3-why-does-this-work--llms-as-reasoning-engines) — chatbot vs RAG vs agent
4. [The Ingredients — What Every Agent Needs](#4-the-ingredients--what-every-agent-needs) — tools, memory, loop controller
5. [Questions You Should Be Asking](#5-questions-you-should-be-asking) — 10 architect questions

### Part II: Build a ReAct Agent from Scratch (S6–S12)
6. [Tools as Functions — The Kitchen Equipment](#6-tools-as-functions--the-kitchen-equipment) — concept & dispatch model
7. [Seed the RAG Knowledge Base](#7-seed-the-rag-knowledge-base) — Horizon Tech Employee Handbook
8. [The ReAct Prompt — The Chef's Playbook](#8-the-react-prompt--the-chefs-playbook) — system prompt deep dive + full agent
9. [Demo: Multi-tool Question](#9-demo-multi-tool-question) — RAG + calculator in one run
10. ["Why Does This Work?" — The Prompt IS the Agent](#10-why-does-this-work--the-prompt-is-the-agent) — prompt-driven programming
11. [Deliberate Mistake: Agent Without Reasoning](#11-deliberate-mistake-agent-without-reasoning) — broken prompt demo
12. [Edge Cases & Decisions](#12-edge-cases--decisions) — failure modes & strategies

### Part III: Advanced Reasoning Patterns (S13–S18)
13. [Chain of Thought Revisited](#13-chain-of-thought-revisited) — structured reasoning within agents
14. [Tree of Thought](#14-tree-of-thought) — explore before committing
15. [Planning Loops & Decomposition](#15-planning-loops--decomposition) — break goals into sub-tasks
16. [ReAct vs CoT vs ToT — When to Use What](#16-react-vs-cot-vs-tot--when-to-use-what) — decision framework
17. [Deliberate Mistake: Over-Planning](#17-deliberate-mistake-over-planning) — when more thinking hurts
18. [Questions You Should Be Asking](#18-questions-you-should-be-asking) — 10 architect questions

### Part IV: Multi-Agent Systems & MCP (S19–S24)
19. [Why Multiple Agents?](#19-why-multiple-agents) — specialization & routing
20. [Build a Multi-Agent Router](#20-build-a-multi-agent-router) — classify → dispatch → merge
21. [Model Context Protocol (MCP)](#21-model-context-protocol-mcp) — the USB-C of agent-tool communication
22. [Build an MCP Server & Client](#22-build-an-mcp-server--client) — hands-on MCP
23. [Framework Landscape](#23-framework-landscape) — LangGraph, CrewAI, AutoGen, raw code
24. [Questions You Should Be Asking](#24-questions-you-should-be-asking) — 10 architect questions

### Part V: Production & Integration (S25–S27 + Wrap-Up)
25. [Production AgentPipeline Class](#25-production-agentpipeline-class) — logging, retries, cost tracking
26. [Evaluation — How Do You Know It Works?](#26-evaluation--how-do-you-know-it-works) — metrics, traces, baselines
27. [Real-World Agent Architectures](#27-real-world-agent-architectures) — 6 products analyzed
- [Self-Check Questions](#self-check-questions)
- [Map Forward](#map-forward)
- [Project Ideas & Cleanup](#project-ideas--cleanup)
- [Glossary](#glossary)

In [ ]:
# ============================================================
# Setup — Install dependencies and define helpers
# ============================================================

# WHY: chromadb for RAG tool's vector database
# WHY: langgraph for framework comparison in Part IV
# WHY: mcp for Model Context Protocol demo in Part IV
import subprocess
import sys

def install(pkg):
    """Install a package if not already available."""
    # WHY: subprocess ensures we install into the running kernel's environment
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# WHY: Install all dependencies upfront so cells run without interruption
# WHY: Map pip names to import names (they differ for some packages)
packages = {
    "chromadb": "chromadb",
    "langgraph": "langgraph",
    "mcp": "mcp",
}

for pkg, import_name in packages.items():
    try:
        __import__(import_name)
    except ImportError:
        install(pkg)

# WHY: Standard library imports for agent infrastructure
import json                         # WHY: Parse LLM responses that include JSON
import re                           # WHY: Regex to extract Action lines from LLM output
import datetime                     # WHY: Date/time tool needs this
import logging                      # WHY: Agent step logging in production cells
import time                         # WHY: Timestamps for step-level latency tracking
import requests                     # WHY: HTTP calls to Ollama API
from dataclasses import dataclass, field   # WHY: Clean data structures for agent state
from typing import Dict, List, Optional, Callable, Any  # WHY: Type hints for clarity

# ============================================================
# Ollama helpers — all LLM calls go through these two functions
# ============================================================

# WHY: Ollama runs models locally — no API keys, no cost, no rate limits
OLLAMA_BASE = "http://localhost:11434"

def ollama_generate(prompt: str, model: str = "mistral") -> str:
    """Send a prompt to Ollama and return the response text.

    Uses /api/generate — the simple completion endpoint.
    Good for single-turn prompts where no conversation history is needed.
    """
    # WHY: /api/generate is Ollama's stateless completion endpoint
    # WHY: stream=False returns the full response in one JSON object
    resp = requests.post(
        f"{OLLAMA_BASE}/api/generate",
        json={"model": model, "prompt": prompt, "stream": False}
    )
    resp.raise_for_status()
    return resp.json()["response"]


def ollama_chat(messages: list, model: str = "mistral") -> str:
    """Send a chat conversation to Ollama and return the assistant's reply.

    Uses /api/chat — supports multi-turn conversation with roles.
    Required for agents because the message history IS the agent's memory.
    """
    # WHY: /api/chat supports system/user/assistant roles for multi-turn conversations
    # WHY: stream=False collects the full response before returning
    resp = requests.post(
        f"{OLLAMA_BASE}/api/chat",
        json={"model": model, "messages": messages, "stream": False}
    )
    resp.raise_for_status()
    return resp.json()["message"]["content"]


# ============================================================
# Verify Ollama is running and mistral is available
# ============================================================
try:
    # WHY: /api/tags lists all locally downloaded models
    r = requests.get(f"{OLLAMA_BASE}/api/tags", timeout=5)
    models = [m["name"] for m in r.json().get("models", [])]
    print(f"\u2713 Ollama is running. Available models: {models}")
    # WHY: Warn early if mistral is missing — every code cell in this notebook needs it
    if not any("mistral" in m for m in models):
        print("\u2717 WARNING: 'mistral' model not found. Run: ollama pull mistral")
    else:
        print("\u2713 mistral model found.")
except requests.ConnectionError:
    # WHY: Clear instructions so the student knows exactly what to do
    print("\u2717 WARNING: Ollama not running. Start it with: ollama serve")
    print("  Then pull the model: ollama pull mistral")

print("\nSetup complete.")

---
# Part I: What Is an Agent?
*The kitchen opens for business.*

---

## 0. You Already Use AI Agents Every Day

Before defining what an agent is, consider three things you may have done this week:

**"Hey Siri, set a timer for 10 minutes."**
Voice input → parse intent → call Timer API → confirm to user.

**Google Maps rerouting around traffic.**
Detect delay → evaluate 3 alternate routes → select fastest → notify driver → update ETA.

**GitHub Copilot autofix.**
Detect failing test → read error + surrounding code → generate a fix → apply the patch → re-run the test.

These are all **agents**. An agent is an AI system that **reasons** about a situation, **chooses** an action, **executes** it, and **observes** the result — then decides what to do next.

> **Misconception:** Agent ≠ chatbot with more features. A chatbot *responds*. An agent *acts*.

---

### The Chef Analogy

This notebook uses a single analogy throughout: **the agent is a chef**.

```
┌─────────────────────────────────────────────────────┐
│                    THE KITCHEN                        │
│                                                       │
│  📋 Order         🧑‍🍳 Chef           🍽️ Plate        │
│  (user query)     (LLM (Large Language Model) + loop)      (final answer)  │
│                                                       │
│       🔪 Knife     🔥 Oven     🧮 Scale              │
│       (calculator) (RAG)      (text analyzer)        │
│                                                       │
│  Chef reads order → picks tool → uses it →            │
│  checks result → picks next tool → ... → plates       │
└─────────────────────────────────────────────────────┘
```

The RAG notebook taught you to build a **librarian** — it finds information. The Multimodal notebook taught you to build a **newsroom** — it perceives the world. Now you build the **chef** — the one who reads the order, decides what to cook, picks the right tools, checks the result, and keeps going until the dish is ready.

---

### Pipeline vs Agent

|  | Pipeline | Agent |
|---|---|---|
| **Control flow** | Fixed sequence | Dynamic, LLM-decided |
| **Tool selection** | Hardcoded | Runtime choice |
| **Error handling** | Crash or skip | Retry, adapt, fall back |
| **Kitchen analogy** | Assembly line (fast food) | Chef (fine dining) |

A pipeline is a conveyor belt: step 1 always feeds step 2 always feeds step 3. An agent is a chef: read the order, decide what to cook, adjust on the fly.

> The librarian found the recipe (RAG). The newsroom gathered ingredients (Multimodal). Now the chef cooks.

## 1. The Agent Loop

Every agent runs the same core loop. The industry calls it **ReAct** (Reason + Act):

```
        ┌──────────┐
        │  THOUGHT  │  ← Chef reads the order / checks progress
        └────┬─────┘
             │
             ▼
        ┌──────────┐
        │  ACTION   │  ← Chef picks a tool and uses it
        └────┬─────┘
             │
             ▼
        ┌──────────┐
        │OBSERVATION│  ← Chef tastes / inspects the result
        └────┬─────┘
             │
             ▼
        ┌──────────┐
        │  REPEAT   │  ← Not done? Go back to THOUGHT
        │  or DONE  │  ← Done? Plate the answer
        └──────────┘
```

Each iteration is one **step**. The agent decides when it is done. The LLM generates the Thought, the framework dispatches the Action, and the tool returns the Observation.

### Chef Parallel

1. **THOUGHT:** Read the order — "Onion soup. Need onions, broth, bread, cheese."
2. **ACTION:** Chop onions.
3. **OBSERVATION:** Onions are caramelized. Broth is cold.
4. **THOUGHT:** Need to heat the broth.
5. **ACTION:** Heat broth on stove.
6. **OBSERVATION:** Broth is simmering. Ready to combine.
7. **THOUGHT:** Combine, add bread and cheese, broil.
8. **ACTION:** Assemble and broil.
9. **OBSERVATION:** Golden, bubbling. Done.
10. **DONE:** Plate and serve.

The chef does not follow a fixed script. Each observation informs the next thought. That is the difference between a pipeline and an agent.

In [ ]:
# ============================================================
# Section 2 — Hello World: Your First Agent (~30 lines)
# ============================================================
# WHY: Build the simplest possible agent to see the Thought/Action/Observation loop
# WHY: One tool (calculator), one question, one loop — nothing else

import re
import requests

# WHY: Only two imports needed — re for parsing LLM output, requests for Ollama HTTP
# WHY: This minimal dependency set is intentional — agents should be lightweight

# WHY: Define exactly one tool — a calculator that evaluates math expressions
# WHY: Using a restricted namespace for eval to prevent arbitrary code execution
def calculator(expr: str) -> str:
    """Evaluate a math expression safely."""
    # WHY: __builtins__={} blocks access to import, open, exec, etc.
    allowed = {"__builtins__": {}}
    return str(eval(expr, allowed))

# WHY: The system prompt teaches the LLM the Thought/Action/Observation format
# WHY: Without this structure, the LLM would just answer directly (no tool use)
SYSTEM_PROMPT = "You are a helpful assistant with access to one tool:\n" \
    "  - calculator(expression): Evaluate a math expression. Example: calculator(25 * 47)\n" \
    "\n" \
    "To use the tool, respond EXACTLY in this format:\n" \
    "Thought: <your reasoning>\n" \
    "Action: calculator(<math expression>)\n" \
    "\n" \
    "When you have the final answer, respond EXACTLY in this format:\n" \
    "Thought: I now have the answer.\n" \
    "Final Answer: <your answer>\n" \
    "\n" \
    "Always start with a Thought."

# WHY: The agent loop — this is the core of every agent
def hello_agent(question: str) -> str:
    """Run a minimal ReAct agent with a single calculator tool."""
    # WHY: Message history is the agent's memory — it accumulates each step
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question}
    ]

    # WHY: max_steps prevents infinite loops if the LLM never says "Final Answer"
    # WHY: 5 steps is generous for a single-tool question; production agents tune this per task
    for step in range(5):
        # WHY: Each call sends the FULL history — the LLM sees all prior steps
        response = ollama_chat(messages, model="mistral")
        print(f"\n--- Step {step + 1} ---")
        print(response)

        # WHY: Check if the agent decided it is done
        if "Final Answer:" in response:
            return response.split("Final Answer:")[-1].strip()

        # WHY: Parse the Action line to find tool name and input
        action_match = re.search(r"Action:\s*calculator\((.+?)\)", response)
        if action_match:
            expr = action_match.group(1).strip()
            # WHY: Execute the tool and feed result back as Observation
            result = calculator(expr)
            observation = f"Observation: {result}"
            print(observation)
            # WHY: Append both the LLM's response and the observation to history
            messages.append({"role": "assistant", "content": response})
            messages.append({"role": "user", "content": observation})
        else:
            # WHY: No valid action found — nudge the agent to use the tool or finish
            messages.append({"role": "assistant", "content": response})
            messages.append({"role": "user", "content": "Please use calculator() or provide a Final Answer."})

    return "Max steps reached."


# --- Run it ---
# WHY: A simple math question that requires the calculator tool
# WHY: 25 * 47 is non-trivial enough that the LLM cannot do it in its head reliably
try:
    answer = hello_agent("What is 25 * 47?")
    print(f"\n{'='*40}")
    print(f"Final: {answer}")
    print(f"{'='*40}")
    print("\nThat's an agent. Thought \u2192 Action \u2192 Observation \u2192 Answer.")
except requests.ConnectionError:
    # WHY: Graceful failure if Ollama is not running
    print("Error: Cannot connect to Ollama. Run: ollama serve")
except Exception as e:
    # WHY: Catch-all so the notebook does not crash on unexpected errors
    print(f"Error (is Ollama running with mistral?): {e}")

## 3. "Why Does This Work?" — LLMs as Reasoning Engines

LLMs are **reasoning engines**, not execution engines. They predict what text should come next. When the system prompt structures the output as Thought/Action/Observation, the LLM "plays the role" of a reasoner — because that is the pattern it was trained to continue.

> The LLM is a **router**. It reads the situation and decides which tool to call. The tool does the actual work.

The LLM never runs the calculator. It outputs the *text* `Action: calculator(25 * 47)`. The framework parses that text and calls the real Python function. The LLM is the decision-maker; the tools are the executors.

---

### Chatbot vs RAG vs Agent

|  | Chatbot | RAG System | Agent |
|---|---|---|---|
| **Input** | Question | Question | Goal |
| **Process** | Generate answer | Retrieve → Generate | Reason → Act → Observe → Repeat |
| **Tools** | None | Vector DB | Multiple, dynamic |
| **Output** | Text | Text + sources | Text + actions taken |
| **Kitchen** | Microwave dinner | Recipe + ingredients | Chef cooking a meal |

A chatbot is a microwave: put food in, get food out. No choices.
A RAG system is a recipe with ingredients: look up the recipe, gather the ingredients, assemble.
An agent is a chef: read the order, decide what to make, pick tools, adjust as you go.

---

> **Architect's Note:** The ReAct pattern (Reason + Act) was introduced by Yao et al. (2022) from Princeton and Google Brain. Their key finding: interleaving reasoning traces with actions improved task success by 20–30% over reasoning-only (chain-of-thought) or acting-only (direct tool use) approaches. The paper tested on HotpotQA, FEVER, and ALFWorld. The insight — reasoning without action is speculation; action without reasoning is blind. ([arXiv:2210.03629](https://arxiv.org/abs/2210.03629))

## 4. The Ingredients — What Every Agent Needs

Every agent, from the hello-world example above to Atlassian Rovo's production agents, requires these five ingredients:

| Ingredient | What It Does | Kitchen Analogy |
|---|---|---|
| **LLM** | Reasons, plans, decides next action | The chef's brain |
| **Tools** | Execute specific actions (math, search, API calls) | Kitchen equipment — knife, oven, scale |
| **System Prompt** | Defines behavior, format, and constraints | The recipe / house rules |
| **Memory** | Tracks conversation history and past actions | The chef's notepad |
| **Loop Controller** | Manages iterations, detects completion, enforces limits | The kitchen timer / head chef |

Remove any one ingredient and the agent breaks:

- **No LLM** → no reasoning. The tools sit idle because nothing decides when to use them.
- **No tools** → no action. The LLM can only talk — it cannot *do* anything.
- **No system prompt** → chaos. The LLM does not know the Thought/Action/Observation format.
- **No memory** → amnesia. The agent repeats itself because it cannot see its prior steps.
- **No loop controller** → infinite loops. The agent runs forever (and burns through tokens).

The next cell builds a **tool registry** — the menu the chef reads from.

In [ ]:
# ============================================================
# Section 4 — Tool Registry Pattern
# ============================================================
# WHY: A tool registry decouples tool definitions from the agent prompt.
# WHY: Adding a new tool automatically updates the system prompt — no manual editing.
# WHY: This is how production agent frameworks (LangChain, LangGraph) work internally.

import datetime
import re

# WHY: Global dict stores tool name -> {description, fn}
# WHY: Using a dict allows O(1) lookup when dispatching actions
TOOLS = {}


def register_tool(name: str, description: str, fn):
    """Register a tool with its name, description, and callable function."""
    # WHY: Store metadata alongside the function for auto-prompt generation
    TOOLS[name] = {"description": description, "fn": fn}


# ============================================================
# Register 3 tools — the chef's basic kitchen equipment
# ============================================================

# WHY: Calculator handles math — restricted eval for safety
register_tool(
    "calculator",
    "Evaluate a math expression. Input: math expression string. Example: calculator(25 * 47)",
    lambda expr: str(eval(expr, {"__builtins__": {}}))
)

# WHY: get_date handles temporal queries ("what day is it?", "is it a weekend?")
register_tool(
    "get_date",
    "Get today's date and day of the week. Input: ignored.",
    lambda _: f"{datetime.date.today()} ({datetime.date.today().strftime('%A')})"
)

# WHY: word_count handles text analysis queries
register_tool(
    "word_count",
    "Count words in the given text. Input: text string.",
    lambda text: str(len(text.split()))
)


def build_system_prompt(tools: dict) -> str:
    """Generate the system prompt from registered tools.

    WHY: Auto-generated prompt ensures tool descriptions always match the registry.
    WHY: When you add/remove a tool, the prompt updates automatically.
    """
    # WHY: Build tool list from registry — single source of truth
    tool_descriptions = "\n".join(
        f"  - {name}({name}_input): {info['description']}"
        for name, info in tools.items()
    )
    # WHY: Structured format teaches the LLM the exact output pattern
    return f"""You are a helpful assistant that can use tools to answer questions.

Available tools:
{tool_descriptions}

To use a tool, respond EXACTLY in this format:
Thought: <your reasoning about what to do>
Action: <tool_name>(<input>)

When you have the final answer, respond EXACTLY in this format:
Thought: I now have the answer.
Final Answer: <your answer>

Rules:
- Always start with a Thought.
- Use one tool at a time.
- Wait for the Observation before your next Thought."""


# WHY: Print the auto-generated prompt so the learner sees what the LLM receives
system_prompt = build_system_prompt(TOOLS)
print("=== Auto-Generated System Prompt ===\n")
print(system_prompt)
print(f"\n=== {len(TOOLS)} tools registered ===")
print(f"Tools: {', '.join(TOOLS.keys())}")


# ============================================================
# Demo: Agent uses the tool registry
# ============================================================
# WHY: This proves the registry works — the agent discovers tools from the prompt
# WHY: No hardcoded tool names in the agent logic; it parses whatever the LLM outputs


def run_agent(question: str, tools: dict, max_steps: int = 5) -> str:
    """Run a simple ReAct agent with the given tools.

    Args:
        question: The user's question or goal.
        tools: Dict of tool_name -> {description, fn}.
        max_steps: Safety limit to prevent infinite loops.

    Returns:
        The agent's final answer as a string.
    """
    # WHY: Build the system prompt from whatever tools are registered right now
    system = build_system_prompt(tools)
    # WHY: Message history = the agent's working memory
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": question}
    ]

    for step in range(max_steps):
        # WHY: Each iteration = one Thought/Action cycle
        response = ollama_chat(messages, model="mistral")
        print(f"\n--- Step {step + 1} ---")
        print(response)

        # WHY: Check if the agent decided it is done
        if "Final Answer:" in response:
            return response.split("Final Answer:")[-1].strip()

        # WHY: Parse the Action line using regex — matches tool_name(input)
        action_match = re.search(r"Action:\s*(\w+)\((.+?)\)", response, re.DOTALL)
        if action_match:
            tool_name = action_match.group(1).strip()
            tool_input = action_match.group(2).strip().strip("\"'")

            if tool_name in tools:
                # WHY: Execute the real Python function and capture result
                try:
                    result = tools[tool_name]["fn"](tool_input)
                except Exception as e:
                    # WHY: Tool errors become Observations — the agent can adapt
                    result = f"Error: {e}"
                observation = f"Observation: {result}"
            else:
                # WHY: Unknown tool — tell the agent so it can pick a valid one
                observation = f"Observation: Error — tool '{tool_name}' not found. Available: {', '.join(tools.keys())}"

            print(observation)
            # WHY: Append both LLM output and observation to build up history
            messages.append({"role": "assistant", "content": response})
            messages.append({"role": "user", "content": observation})
        else:
            # WHY: No action parsed and no final answer — nudge the agent
            messages.append({"role": "assistant", "content": response})
            messages.append({"role": "user", "content": "Please use a tool or provide a Final Answer."})

    return "Max steps reached without final answer."


# --- Demo: Ask a date question ---
# WHY: This question requires the get_date tool, proving the agent discovers it from the prompt
try:
    answer = run_agent("What is today's date?", TOOLS)
    print(f"\nFinal: {answer}")
except requests.ConnectionError:
    # WHY: Graceful failure if Ollama is not running
    print("Error: Cannot connect to Ollama. Run: ollama serve")
except Exception as e:
    # WHY: Catch-all for unexpected errors
    print(f"Error (is Ollama running with mistral?): {e}")

## 5. Questions You Should Be Asking

Pause and think like an architect. After building two agents (hello-world and tool-registry), these questions should be forming:

1. **What happens when the LLM picks the wrong tool?** — It gets an unhelpful Observation and wastes a step. How do you detect and recover from this?
2. **How do you prevent infinite loops?** — Max steps is the blunt instrument. Can you detect *repeated* actions (loop detection)?
3. **What does each agent step cost?** — Every step = one LLM call. Tokens in (full history) + tokens out (response). Step 5 sends *all* prior steps as input.
4. **How do you handle tool failures?** — API down, timeout, malformed input. The agent needs an error Observation, not a crash.
5. **What if the LLM generates invalid Action syntax?** — Regex fails to parse. The agent needs a fallback prompt: "Please re-format your Action."
6. **How do you log and debug agent runs?** — Step-level logs with timestamps, token counts, tool calls, and observations. Without this, agents are black boxes.
7. **What is the latency per step?** — LLM inference time + tool execution time. A 5-step agent with 2s per LLM call = 10s minimum.
8. **How do you evaluate if the agent answered correctly?** — Unit tests for tools. End-to-end tests for full agent runs. Ground truth datasets.
9. **Can agents call other agents as tools?** — Yes. This is multi-agent orchestration (Part IV of this notebook).
10. **How do you version prompts when the prompt IS the logic?** — Prompt changes = behavior changes. Treat prompts like code: version control, review, test.

We answer most of these by the end of this notebook.

---
# Part II: Build a ReAct Agent from Scratch
*The chef learns to cook a real meal.*

---

## 6. Tools as Functions — The Kitchen Equipment

A tool is a **function with metadata**. The LLM never executes the function — it outputs a structured request ("use the calculator with input 25*47"), and the **framework** dispatches the call.

```
LLM outputs:    "Action: calculator(25 * 47)"
                     ↓
Framework parses: tool_name = "calculator",  input = "25 * 47"
                     ↓
Framework calls:  TOOLS["calculator"]["fn"]("25 * 47")
                     ↓
Python returns:   "1175"
                     ↓
Framework feeds:  "Observation: 1175"  →  back to LLM
```

In kitchen terms: the chef says **"oven, 350 degrees, 20 minutes."** The chef does not *become* the oven — the chef *operates* it. The oven does the work. The chef checks the result.

This separation matters because:
- **Tools can be tested independently.** Write unit tests for each tool function.
- **Tools can be swapped.** Replace the calculator with Wolfram Alpha — the agent does not change.
- **Tools can fail safely.** An exception becomes an error Observation, not a crash.
- **Tools can be metered.** Track how often each tool is called, how long it takes, what it costs.

The next cell builds four production-quality tools.

In [ ]:
# ============================================================
# Section 6 — Build 4 Production-Quality Tools
# ============================================================
# WHY: These are the kitchen equipment the chef (agent) will use
# WHY: Each tool has error handling, docstrings, and metadata for auto-prompting

import datetime
import math
import chromadb


# WHY: Reset the global tool registry so Part II starts clean
TOOLS = {}


def register_tool(name: str, description: str, fn):
    """Register a tool for agent use."""
    # WHY: Same pattern as before — name, description, callable
    TOOLS[name] = {"description": description, "fn": fn}


# ============================================================
# Tool 1: Calculator (restricted eval for safety)
# ============================================================
def tool_calculator(expr: str) -> str:
    """Evaluate a math expression using restricted Python eval.

    WHY: Agents need math. LLMs are bad at arithmetic.
    WHY: __builtins__={} prevents import, open, exec, etc.
    WHY: Allowed names give access to common math operations.
    """
    # WHY: Whitelist only safe math functions — no file I/O, no imports
    allowed_names = {
        "__builtins__": {},
        "abs": abs, "round": round, "min": min, "max": max,
        "sum": sum, "len": len, "pow": pow, "int": int, "float": float,
        "sqrt": math.sqrt, "pi": math.pi, "e": math.e,
    }
    try:
        # WHY: eval with restricted namespace is safe for math-only expressions
        result = eval(expr, allowed_names)
        return str(result)
    except Exception as e:
        # WHY: Return error as string so the agent sees it as an Observation
        return f"Calculator error: {e}"

# WHY: Register with clear description so the LLM knows when to use it
register_tool(
    "calculator",
    "Evaluate a math expression. Input: a math expression string. Example: calculator(25 * 47 + 3)",
    tool_calculator
)


# ============================================================
# Tool 2: RAG Lookup (ChromaDB vector search)
# ============================================================
# WHY: The RAG tool gives the agent access to a knowledge base
# WHY: ChromaDB handles embedding + similarity search in one call
# WHY: Initialize the collection here; seed it in the next cell

chroma_client = chromadb.Client()
# WHY: get_or_create prevents errors if this cell is re-run
handbook_collection = chroma_client.get_or_create_collection(
    name="horizon_handbook",
    metadata={"hnsw:space": "cosine"}  # WHY: cosine similarity is standard for text
)

def tool_rag_lookup(query: str) -> str:
    """Search the Horizon Tech Employee Handbook for relevant information.

    WHY: Returns top 3 most relevant chunks so the agent has context to answer.
    WHY: If the collection is empty, returns a helpful error message.
    """
    # WHY: Check if knowledge base has been seeded
    if handbook_collection.count() == 0:
        return "Error: Knowledge base is empty. Run the seeding cell first."
    # WHY: n_results=3 balances relevance vs. context window usage
    results = handbook_collection.query(query_texts=[query], n_results=3)
    # WHY: Join the documents with separators for readable Observations
    docs = results["documents"][0]
    return "\n---\n".join(docs)

# WHY: Description tells the LLM this searches company policies
register_tool(
    "rag_lookup",
    "Search the Horizon Tech Employee Handbook. Input: a search query string. Example: rag_lookup(PTO policy)",
    tool_rag_lookup
)


# ============================================================
# Tool 3: Get Date/Time
# ============================================================
def tool_get_datetime(_: str) -> str:
    """Return current date, time, and day of week.

    WHY: Agents need temporal awareness for scheduling, deadline, and policy questions.
    WHY: Input parameter is ignored — included for consistent tool interface.
    """
    # WHY: strftime gives human-readable date + day name
    now = datetime.datetime.now()
    return f"{now.strftime('%Y-%m-%d %H:%M:%S')} ({now.strftime('%A')})"

# WHY: Clear description so the LLM uses this for time/date questions
register_tool(
    "get_datetime",
    "Get the current date, time, and day of week. Input: ignored. Example: get_datetime(now)",
    tool_get_datetime
)


# ============================================================
# Tool 4: Text Analyzer
# ============================================================
def tool_text_analyzer(text: str) -> str:
    """Analyze text and return word count, sentence count, and avg word length.

    WHY: Useful for content analysis, summarization decisions, and format checks.
    WHY: Returns structured data so the agent can use specific values.
    """
    # WHY: Split on whitespace for word count
    words = text.split()
    word_count = len(words)
    # WHY: Split on sentence-ending punctuation for sentence count
    sentences = [s.strip() for s in re.split(r'[.!?]+', text) if s.strip()]
    sentence_count = len(sentences)
    # WHY: Average word length helps gauge text complexity
    avg_word_len = round(sum(len(w) for w in words) / max(word_count, 1), 1)
    return f"Words: {word_count}, Sentences: {sentence_count}, Avg word length: {avg_word_len} chars"

# WHY: Register with description that tells the LLM about all three metrics
register_tool(
    "text_analyzer",
    "Analyze text: returns word count, sentence count, and average word length. Input: text string.",
    tool_text_analyzer
)


# ============================================================
# Print the tool registry — the menu the chef reads from
# ============================================================
print("=" * 50)
print("TOOL REGISTRY — The Chef's Kitchen Equipment")
print("=" * 50)
# WHY: Show what the agent will see in its system prompt
for name, info in TOOLS.items():
    print(f"\n  {name}:")
    print(f"    {info['description']}")
print(f"\n{len(TOOLS)} tools registered and ready.")

In [ ]:
# ============================================================
# Section 7 — Seed the Agent's Pantry: Horizon Tech Handbook
# ============================================================
# WHY: The RAG tool needs a knowledge base to search
# WHY: Same corpus from the RAG notebook — Horizon Tech Employee Handbook
# WHY: These chunks represent what a real employee handbook contains

# WHY: Each chunk is a self-contained policy section, labeled clearly
# WHY: Chunk IDs are meaningful (not random) so we can debug retrieval issues
# WHY: 10 chunks cover the major HR policy areas an agent would need to answer about
# WHY: Chunk length is 50-80 words each — fits in a single embedding comfortably
# WHY: Each chunk starts with "Horizon Tech [Topic]:" for consistent formatting
# WHY: This is the same corpus from the RAG notebook for continuity
handbook_chunks = [
    # --- Chunk 1 of 10 ---
    # WHY: PTO policy — common agent question for HR chatbots
    # WHY: Contains specific numbers (15, 20, 25 days) the agent may need to calculate with
    {
        "id": "pto-policy",
        "text": (
            "Horizon Tech PTO Policy: New employees receive 15 PTO days per year, "
            "accruing at 1.25 days per month starting from their hire date. After 3 years "
            "of service, PTO increases to 20 days per year. After 7 years, it increases to "
            "25 days. Unused PTO carries over up to a maximum of 5 days. PTO requests must "
            "be submitted at least 2 weeks in advance for periods longer than 3 consecutive days."
        ),
    },
    # --- Chunk 2 of 10 ---
    # WHY: Health insurance — tests agent's ability to compare three plan options
    {
        "id": "health-insurance",
        "text": (
            "Horizon Tech Health Insurance: The company offers three health plans — Basic, "
            "Standard, and Premium. Basic covers 70% of medical costs with a $2,000 deductible. "
            "Standard covers 85% with a $1,000 deductible. Premium covers 95% with a $500 "
            "deductible. The company pays 80% of the premium for all plans. Dental and vision "
            "are included in Standard and Premium plans. Enrollment opens annually in November."
        ),
    },
    # --- Chunk 3 of 10 ---
    # WHY: 401k — contains math the agent may need the calculator for (matching formula)
    {
        "id": "401k-matching",
        "text": (
            "Horizon Tech 401(k) Plan: The company matches 100% of employee contributions up "
            "to 4% of salary, and 50% of contributions between 4% and 6%. This means an "
            "employee contributing 6% of their salary receives an effective 5% company match. "
            "Vesting schedule: 25% after 1 year, 50% after 2 years, 75% after 3 years, 100% "
            "after 4 years. Employees can enroll on their first day."
        ),
    },
    # --- Chunk 4 of 10 ---
    # WHY: Remote work — common question in modern workplaces
    {
        "id": "remote-work",
        "text": (
            "Horizon Tech Remote Work Policy: Employees may work remotely up to 3 days per week "
            "with manager approval. Fully remote arrangements require VP-level approval and are "
            "reviewed quarterly. All remote employees must be available during core hours (10 AM "
            "to 3 PM in their local time zone). The company provides a $500 annual home office "
            "stipend for equipment and internet costs."
        ),
    },
    # --- Chunk 5 of 10 ---
    # WHY: Performance reviews — includes numeric ratings (tests precision)
    {
        "id": "performance-reviews",
        "text": (
            "Horizon Tech Performance Reviews: Reviews are conducted semi-annually in June and "
            "December. Each review includes self-assessment, peer feedback (minimum 3 peers), "
            "and manager evaluation. Performance is rated on a 5-point scale: 1 (Below "
            "Expectations), 2 (Approaching), 3 (Meeting), 4 (Exceeding), 5 (Exceptional). "
            "Ratings of 4 or 5 qualify for a merit increase of 3-8%. Ratings below 3 trigger "
            "a Performance Improvement Plan (PIP) with 90-day goals."
        ),
    },
    # --- Chunk 6 of 10 ---
    # WHY: Parental leave — primary vs secondary caregiver distinction
    {
        "id": "parental-leave",
        "text": (
            "Horizon Tech Parental Leave: Primary caregivers receive 16 weeks of fully paid "
            "parental leave. Secondary caregivers receive 6 weeks of fully paid leave. Leave "
            "may be taken within the first 12 months after birth or adoption. An additional 4 "
            "weeks of unpaid leave is available upon request. Employees returning from parental "
            "leave are guaranteed their same position or an equivalent role."
        ),
    },
    # --- Chunk 7 of 10 ---
    # WHY: Professional development — used in the multi-tool demo (budget + math)
    {
        "id": "professional-development",
        "text": (
            "Horizon Tech Professional Development: Each employee receives a $3,000 annual "
            "learning budget for conferences, courses, certifications, and books. An additional "
            "$1,500 is available for cloud certification exams (AWS, GCP, Azure). Employees may "
            "use up to 5 business days per year for professional development activities. Budget "
            "resets on January 1 and does not carry over."
        ),
    },
    # --- Chunk 8 of 10 ---
    # WHY: Office locations — used in the edge case demo (broken tool fallback)
    {
        "id": "office-locations",
        "text": (
            "Horizon Tech Office Locations: Headquarters is in Austin, TX (500 employees). "
            "Regional offices in San Francisco, CA (120 employees), New York, NY (80 employees), "
            "and Chicago, IL (60 employees). International offices in London, UK (40 employees) "
            "and Bangalore, India (200 employees). All offices provide free lunch on Wednesdays, "
            "24/7 building access, and on-site gym facilities."
        ),
    },
    # --- Chunk 9 of 10 ---
    # WHY: Equity compensation — level-based grants with vesting schedules
    {
        "id": "equity-compensation",
        "text": (
            "Horizon Tech Equity Compensation: New hires at the Senior Engineer level and above "
            "receive equity grants (RSUs). Standard grant for Senior Engineers is $50,000 in RSUs "
            "vesting over 4 years with a 1-year cliff. Staff Engineers receive $100,000 and "
            "Principal Engineers receive $200,000. Refresh grants are awarded annually based on "
            "performance ratings. Equity grants are subject to board approval."
        ),
    },
    # --- Chunk 10 of 10 ---
    # WHY: Expense policy — includes per-diem caps the agent may need to calculate with
    {
        "id": "expense-policy",
        "text": (
            "Horizon Tech Expense Policy: Business travel requires pre-approval for expenses "
            "over $500. Hotel reimbursement caps at $250/night (domestic) and $350/night "
            "(international). Meals are reimbursed up to $75/day. Economy class flights for "
            "domestic travel; business class for international flights over 6 hours. All "
            "expenses must be submitted within 30 days with receipts via the Concur system."
        ),
    },
]

# ============================================================
# Load chunks into ChromaDB
# ============================================================
# WHY: Batch add is more efficient than one-at-a-time for ChromaDB
# WHY: ChromaDB auto-generates embeddings using its default model
# WHY: The collection was created in the previous cell (get_or_create)
# WHY: No metadata parameter needed — ChromaDB stores the text as both document and embedding source
handbook_collection.add(
    # WHY: Meaningful IDs allow us to verify which chunk was retrieved
    ids=[chunk["id"] for chunk in handbook_chunks],
    # WHY: documents are auto-embedded by ChromaDB's default embedding function
    documents=[chunk["text"] for chunk in handbook_chunks],
)

# ============================================================
# Verify the knowledge base is populated and working
# ============================================================
# WHY: Always verify after seeding — silent failures are common with vector DBs
print(f"Knowledge base seeded: {handbook_collection.count()} chunks loaded.")
# WHY: Print all topic IDs so the learner sees what is available
# WHY: These IDs double as documentation — you can tell at a glance what the KB covers
print(f"Topics: {', '.join(c['id'] for c in handbook_chunks)}")

# WHY: Quick sanity check — search for "PTO" and confirm the right chunk comes back
# WHY: If this returns the wrong chunk, the embeddings or data are broken
# WHY: n_results=1 returns only the top match — sufficient for a sanity check
test_results = handbook_collection.query(query_texts=["PTO vacation days"], n_results=1)
print(f"\nSanity check — query 'PTO vacation days':")
print(f"  Top result: {test_results['ids'][0][0]}")
# WHY: Preview the first 100 chars so the learner can visually verify relevance
print(f"  Preview: {test_results['documents'][0][0][:100]}...")

## 8. The ReAct Prompt — The Chef's Playbook

The system prompt is the single most important piece of the agent. It defines:

1. **Role** — who the agent is and how it behaves
2. **Available tools** — auto-generated from the tool registry
3. **Response format** — the exact Thought/Action/Observation template
4. **Constraints** — one tool per step, always think first, wait for observations
5. **Stop condition** — when to emit "Final Answer" and stop the loop

Here is the full template the next cell uses:

```
You are a helpful assistant that solves problems step by step using tools.

Available tools:
  - calculator(expression): Evaluate a math expression.
  - rag_lookup(query): Search the Horizon Tech Employee Handbook.
  - get_datetime(input): Get the current date and time.
  - text_analyzer(text): Analyze text for word count and stats.

To use a tool, respond in this EXACT format:
  Thought: <reason about what to do next>
  Action: <tool_name>(<input>)

After receiving an Observation, continue with another Thought.

When you have the final answer:
  Thought: I now have enough information to answer.
  Final Answer: <your complete answer>

Rules:
  - Always start with a Thought.
  - Use exactly one tool per step.
  - Wait for the Observation before your next Thought.
  - If a tool returns an error, reason about it and try a different approach.
  - Do not guess — use tools to get facts.
```

Notice: the **tool descriptions section is auto-generated** from `build_system_prompt()`. Adding a new tool to the registry automatically adds it to this prompt. No manual editing.

> In kitchen terms: this prompt is the **house rules** posted on the kitchen wall. Every chef (LLM call) reads them before starting work.

In [ ]:
# ============================================================
# Section 8 — Full ReAct Agent (Production-Grade)
# ============================================================
# WHY: This is the real agent — proper parsing, error handling, logging, memory
# WHY: Everything before this was warm-up; this is the version you would deploy
# WHY: All subsequent demos (multi-tool, broken prompt, edge cases) use this function

# WHY: re for Action parsing, time for latency tracking, requests for Ollama HTTP calls
import re
import time
import requests


def build_react_prompt(tools: dict) -> str:
    """Build the full ReAct system prompt from registered tools.

    WHY: Auto-generation ensures the prompt is always in sync with the registry.
    WHY: The format section is carefully structured — LLMs are sensitive to formatting.
    """
    # WHY: Build tool descriptions from the registry
    tool_lines = "\n".join(
        f"  - {name}({name}_input): {info['description']}"
        for name, info in tools.items()
    )
    # WHY: Each section of this prompt serves a specific purpose:
    #   1. Role definition — sets the agent's identity
    #   2. Tool list — auto-generated from registry
    #   3. Format — teaches Thought/Action structure
    #   4. Rules — enforces one-tool-per-step discipline
    #   5. Stop condition — tells the LLM when to stop looping
    return (
        "You are a helpful assistant that solves problems step by step using tools.\n"
        "\n"
        "Available tools:\n"
        f"{tool_lines}\n"
        "\n"
        # WHY: EXACT format instruction is critical — LLMs follow explicit patterns better
        "To use a tool, respond in this EXACT format:\n"
        "Thought: <reason about what to do next>\n"
        "Action: <tool_name>(<input>)\n"
        "\n"
        "After receiving an Observation, continue with another Thought.\n"
        "\n"
        # WHY: Explicit stop condition prevents the agent from looping forever
        "When you have enough information to answer, respond EXACTLY:\n"
        "Thought: I now have enough information to answer.\n"
        "Final Answer: <your complete answer>\n"
        "\n"
        # WHY: Rules section constrains LLM behavior to prevent common failure modes
        "Rules:\n"
        "- Always start with a Thought.\n"
        "- Use exactly one tool per step.\n"
        "- Wait for the Observation before continuing.\n"
        "- If a tool returns an error, reason about it and try a different approach.\n"
        "- Do not guess when a tool can give you the facts."
    )


def react_agent(question: str, tools: dict, max_steps: int = 10, verbose: bool = True) -> str:
    """Run a ReAct agent: Thought -> Action -> Observation -> ... -> Final Answer.

    Args:
        question: The user's question or goal.
        tools: Dict of tool_name -> {description, fn}.
        max_steps: Maximum number of Thought/Action iterations.
        verbose: If True, print each step in detail.

    Returns:
        The agent's final answer, or a timeout message.

    WHY: This function is the core of the notebook. Every pattern after this builds on it.
    """
    # WHY: Build system prompt from current tool registry
    system = build_react_prompt(tools)
    # WHY: Message history accumulates — this IS the agent's memory
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": question}
    ]

    # WHY: Track steps for logging and debugging
    step_log = []

    # WHY: 1-indexed loop for human-readable step numbers in output
    for step in range(1, max_steps + 1):
        # WHY: Time each step to expose latency bottlenecks
        step_start = time.time()

        # WHY: Send full history to LLM — it needs all prior steps to reason correctly
        try:
            response = ollama_chat(messages, model="mistral")
        except requests.ConnectionError:
            return "Error: Cannot connect to Ollama. Run: ollama serve"
        except Exception as e:
            return f"Error during LLM call: {e}"

        # WHY: Measure wall-clock time for each step (LLM call + overhead)
        step_duration = time.time() - step_start

        # WHY: Log every step with timing for debugging and cost analysis
        # WHY: In production, this log would include token counts and cost estimates
        step_entry = {
            "step": step,
            "response": response[:200],  # WHY: Truncate for readable logs
            "duration_s": round(step_duration, 2),
        }

        # WHY: Verbose mode shows each step so the learner can follow the agent's reasoning
        if verbose:
            print(f"\n{'='*50}")
            print(f"Step {step} ({step_duration:.1f}s)")
            print(f"{'='*50}")
            print(response)

        # WHY: Check if agent decided it is done — "Final Answer:" is the stop signal
        # WHY: Split on last occurrence in case "Final Answer:" appears in reasoning too
        if "Final Answer:" in response:
            final = response.split("Final Answer:")[-1].strip()
            step_entry["action"] = "DONE"
            step_log.append(step_entry)
            # WHY: Print summary stats so the learner sees total cost of the agent run
            if verbose:
                print(f"\n{'='*50}")
                print(f"Agent finished in {step} steps.")
                print(f"Total time: {sum(s['duration_s'] for s in step_log):.1f}s")
            return final

        # WHY: Parse Action line — matches patterns like: Action: tool_name(input)
        # WHY: re.DOTALL allows input to contain newlines (e.g., multi-line text)
        # WHY: Group 1 = tool name, Group 2 = tool input
        action_match = re.search(r"Action:\s*(\w+)\((.+?)\)", response, re.DOTALL)
        if action_match:
            # WHY: Strip whitespace and quotes from parsed values
            tool_name = action_match.group(1).strip()
            tool_input = action_match.group(2).strip().strip("\"'")

            # WHY: Check tool exists before calling — LLMs sometimes hallucinate tool names
            if tool_name in tools:
                # WHY: Execute the tool and capture result (or error)
                try:
                    tool_start = time.time()
                    result = tools[tool_name]["fn"](tool_input)
                    tool_time = time.time() - tool_start
                    observation = f"Observation: {result}"
                    step_entry["action"] = f"{tool_name}({tool_input[:50]})"
                    step_entry["tool_time_s"] = round(tool_time, 3)
                except Exception as e:
                    # WHY: Tool errors become Observations — agent can adapt
                    observation = f"Observation: Tool error — {e}"
                    step_entry["action"] = f"{tool_name} (ERROR)"
            else:
                # WHY: Unknown tool — give the agent the list of valid tools
                observation = (
                    f"Observation: Error — tool '{tool_name}' not found. "
                    f"Available tools: {', '.join(tools.keys())}"
                )
                step_entry["action"] = f"UNKNOWN({tool_name})"

            # WHY: Show the observation in verbose mode for step-by-step following
            if verbose:
                print(f"\n  >> {observation}")

            # WHY: Append LLM response + Observation to history for next iteration
            # WHY: The Observation goes as a "user" message so the LLM processes it
            messages.append({"role": "assistant", "content": response})
            messages.append({"role": "user", "content": observation})
        else:
            # WHY: LLM did not produce a parseable Action and did not say Final Answer
            # WHY: Nudge it to follow the format
            messages.append({"role": "assistant", "content": response})
            messages.append({"role": "user", "content": (
                "I could not parse an Action from your response. "
                "Please respond with either:\n"
                "  Action: tool_name(input)\n"
                "or:\n"
                "  Final Answer: <your answer>"
            )})
            step_entry["action"] = "PARSE_FAIL"

        step_log.append(step_entry)

    # WHY: Safety net — if max_steps reached, return what we have
    # WHY: In production, this would trigger an alert or escalation
    return f"Agent stopped after {max_steps} steps without a Final Answer."


# WHY: Confirm the function is available for subsequent cells
print("react_agent() defined. Ready to use.")
# WHY: Show registered tools so the learner knows what the agent can do
print(f"Tools available: {', '.join(TOOLS.keys())}")

In [ ]:
# ============================================================
# Section 9 — Demo: Multi-Tool Question
# ============================================================
# WHY: This question requires TWO tools — proving the agent can chain actions
# WHY: Step 1: RAG lookup to find PTO days
# WHY: Step 2: Calculator to compute percentage of 260 working days

# WHY: The question is designed to be impossible without both tools
question = (
    "How many PTO days does a new employee get at Horizon Tech, "
    "and what percentage of 260 working days is that?"
)

# WHY: Print the question and expected flow so the learner can trace the agent's work
print("Question:", question)
print("\nExpected agent behavior:")
print("  1. Use rag_lookup to find PTO policy")
print("  2. Use calculator to compute percentage (15/260 * 100)")
print("  3. Combine both results into a Final Answer")
print()

# WHY: Wrap in try/except so the notebook does not crash if Ollama is down
# WHY: max_steps=8 gives the agent headroom for multi-tool chaining
try:
    answer = react_agent(question, TOOLS, max_steps=8)
    # WHY: Visual separators make the final answer stand out in notebook output
    print(f"\n{'='*50}")
    print(f"FINAL ANSWER: {answer}")
    print(f"{'='*50}")
except requests.ConnectionError:
    # WHY: Clear instructions for the student
    print("Error: Cannot connect to Ollama. Run: ollama serve")
except Exception as e:
    # WHY: Catch-all for unexpected errors
    print(f"Error: {e}")

## 10. "Why Does This Work?" — The Prompt IS the Agent

The system prompt is not configuration. It **is** the agent's personality and behavior. Change the prompt, change the agent. Same LLM, same tools, different prompt → completely different behavior.

This is **prompt-driven programming**:
- **Traditional software:** logic lives in code (if/else, loops, functions).
- **Agent software:** logic lives in the prompt ("always think first," "use one tool at a time," "if a tool errors, try a different approach").

The code framework (the `react_agent()` function) is just plumbing — it parses the LLM's text, dispatches tool calls, and feeds results back. The *intelligence* lives in the prompt.

This has profound implications:
- **Debugging** means reading the prompt, not stepping through code.
- **Versioning** means tracking prompt changes — a one-word change can alter behavior dramatically.
- **Testing** means running the same question with different prompts and comparing outputs.
- **Specialization** means writing different prompts for different tasks — same model, different behavior.

---

> **Architect's Note:** Atlassian's Rovo system uses different prompts for the same underlying LLM to create distinct agent personalities: a brainstorm agent (creative, expansive), a tool Q&A agent (precise, retrieval-focused), and a deep reasoning agent (analytical, multi-step). Same model. Different prompts. Completely different behavior. The prompt is the agent's DNA. (Source: [Atlassian Engineering Blog — Building Rovo Agents](https://www.atlassian.com/engineering/building-rovo-agents))

## 11. Deliberate Mistake: Agent Without Reasoning

What happens when you remove "Thought:" from the prompt?

The agent still works... sort of. Without the reasoning step, the LLM jumps straight to action. It often:
- **Picks the wrong tool first** because it did not reason about which tool matches the question.
- **Makes unnecessary calls** — calling tools it does not need.
- **Gets confused by multi-step problems** — without thinking, it forgets the overall goal.
- **Takes more steps** to reach the answer (or hits the max-step limit).

This is the difference between a chef who reads the order before cooking and one who just starts grabbing pans.

The next cell demonstrates this with a side-by-side comparison.

In [ ]:
# ============================================================
# Section 11 — Broken Prompt Demo: With vs Without Reasoning
# ============================================================
# WHY: Demonstrates why the "Thought:" step matters
# WHY: Same question, same tools — only difference is the prompt

# WHY: re for Action parsing, time for step timing, requests for Ollama calls
import re
import time
import requests


def build_broken_prompt(tools: dict) -> str:
    """Build a system prompt WITHOUT the Thought step."""
    # WHY: This intentionally removes reasoning to show its impact
    # WHY: The agent can only say Action or Final Answer — no thinking allowed
    # WHY: Same tool descriptions as the full prompt, but no Thought instruction
    tool_lines = "\n".join(
        f"  - {name}({name}_input): {info['description']}"
        for name, info in tools.items()
    )
    # WHY: Notice what is MISSING: no "Thought:" instruction, no "reason about" language
    # WHY: The LLM will jump straight to actions without any reasoning step
    return (
        "You are a helpful assistant that uses tools to answer questions.\n"
        "\n"
        "Available tools:\n"
        f"{tool_lines}\n"
        "\n"
        # WHY: Only Action format — no Thought format at all
        "To use a tool:\n"
        "Action: <tool_name>(<input>)\n"
        "\n"
        # WHY: Only Final Answer — no reasoning before answering
        "When done:\n"
        "Final Answer: <your answer>\n"
        "\n"
        "Use tools to get facts. Do not guess."
    )


def run_comparison(question: str, tools: dict, max_steps: int = 8):
    """Run the same question with broken prompt and full ReAct prompt.

    WHY: Side-by-side comparison makes the impact of reasoning visible.
    WHY: Count steps to show efficiency difference.
    """
    # --- Run 1: Broken prompt (no Thought) ---
    # WHY: Run the broken version FIRST so the learner sees the contrast
    print("=" * 60)
    print("RUN 1: BROKEN PROMPT (no Thought step)")
    print("=" * 60)

    # WHY: Build the crippled prompt — no reasoning instructions
    broken_system = build_broken_prompt(tools)
    # WHY: Same message structure as the real agent
    messages = [
        {"role": "system", "content": broken_system},
        {"role": "user", "content": question}
    ]
    # WHY: Track step count for comparison summary
    broken_steps = 0

    for step in range(1, max_steps + 1):
        # WHY: Each step is one LLM call — same cost regardless of prompt quality
        response = ollama_chat(messages, model="mistral")
        broken_steps = step
        print(f"\n--- Step {step} ---")
        # WHY: Truncate long responses for readability in notebook output
        print(response[:300] + ("..." if len(response) > 300 else ""))

        # WHY: Check for completion — same stop condition as the real agent
        if "Final Answer:" in response:
            print(f"\n>> Finished in {step} steps")
            break

        # WHY: Same parsing logic as the real agent — only the prompt differs
        action_match = re.search(r"Action:\s*(\w+)\((.+?)\)", response, re.DOTALL)
        if action_match:
            # WHY: Extract tool name and input from the LLM's response
            tool_name = action_match.group(1).strip()
            tool_input = action_match.group(2).strip().strip("\"'")
            if tool_name in tools:
                try:
                    # WHY: Execute the tool normally
                    result = tools[tool_name]["fn"](tool_input)
                    observation = f"Observation: {result}"
                except Exception as e:
                    # WHY: Tool errors become Observations (same as production agent)
                    observation = f"Observation: Error — {e}"
            else:
                # WHY: Unknown tool — common with no-reasoning agents
                observation = f"Observation: Error — tool '{tool_name}' not found."
            # WHY: Show the observation so the learner can follow the agent's journey
            print(f"  >> {observation[:200]}")
            # WHY: Append to history for multi-turn conversation
            messages.append({"role": "assistant", "content": response})
            messages.append({"role": "user", "content": observation})
        else:
            # WHY: No parseable action — common failure mode without reasoning
            messages.append({"role": "assistant", "content": response})
            messages.append({"role": "user", "content": "Use Action: tool(input) or Final Answer:"})
    else:
        # WHY: for/else fires when loop completes without break (no Final Answer)
        print(f"\n>> Max steps ({max_steps}) reached without answer.")

    # --- Run 2: Full ReAct prompt ---
    # WHY: Now run the SAME question with the full prompt for direct comparison
    print(f"\n\n{'=' * 60}")
    print("RUN 2: FULL ReAct PROMPT (with Thought step)")
    print("=" * 60)

    # WHY: Use the production react_agent — same code, better prompt
    try:
        answer = react_agent(question, tools, max_steps=max_steps)
        print(f"\nFinal: {answer}")
    except Exception as e:
        print(f"Error: {e}")

    # WHY: Print summary so the comparison is clear at a glance
    print(f"\n{'=' * 60}")
    print("COMPARISON SUMMARY")
    print(f"{'=' * 60}")
    print(f"  Without Thought: {broken_steps} steps (may have wrong answer)")
    print(f"  With Thought:    see above (typically fewer steps, correct answer)")
    print(f"\n  Reasoning = fewer steps + correct answers + better tool selection.")


# --- Run the comparison ---
# WHY: Use a multi-step question to amplify the difference
comparison_question = (
    "What is the annual professional development budget at Horizon Tech, "
    "and how many $200 courses could an employee take with it?"
)

try:
    run_comparison(comparison_question, TOOLS)
except requests.ConnectionError:
    # WHY: Graceful failure if Ollama is not running
    print("Error: Cannot connect to Ollama. Run: ollama serve")
except Exception as e:
    # WHY: Catch-all for unexpected errors
    print(f"Error: {e}")

## 12. Edge Cases & Decisions

Building a working agent is step one. Making it **reliable** is where the real engineering starts. Every edge case below has happened in production systems:

| Edge Case | What Happens | Strategy |
|---|---|---|
| **Tool returns error** | Agent receives error as Observation | Catch exceptions, return error string as Observation. Agent can reason about it and retry or try a different tool. |
| **Malformed LLM output** | Regex cannot parse Action line | Re-prompt: "I could not parse your Action. Please use the format: Action: tool_name(input)" |
| **Infinite loop** | Agent repeats the same Action/Observation | Track last N actions. If duplicate detected, inject: "You already tried this. Try a different approach." |
| **Contradictory observations** | Tool A says X, Tool B says Y | Do not auto-resolve. Prompt agent: "These results contradict. Reason about which is more reliable." |
| **Tool timeout** | Agent hangs waiting for tool response | Set per-tool timeouts. On timeout, return: "Observation: Tool timed out after N seconds." |
| **No tool matches** | Agent hallucinates a tool name | Return: "Error — tool 'X' not found. Available: ..." The agent sees its options and self-corrects. |
| **Excessive steps** | Agent wanders without converging | Max-step limit (hard stop) + cost tracking (warn when tokens exceed budget). |
| **Prompt injection via tool** | Malicious content in tool output | Sanitize Observations before feeding back. Never execute code from Observations. |

### The 80/20 Rule of Agent Reliability

The first version of any agent handles the happy path. That covers about 60% of real usage. The next 20% comes from handling the six edge cases above. The final 20% — ambiguous queries, adversarial inputs, cascading failures — is where you need evaluation pipelines (Section 26).

> In kitchen terms: a chef who can cook is useful. A chef who can cook, handle a broken oven, adapt when an ingredient runs out, and still plate on time — that is a *professional*.

In [ ]:
# ============================================================
# Section 12 — Edge Case Demo: Broken Tool Recovery
# ============================================================
# WHY: Demonstrates that agents can handle tool failures gracefully
# WHY: A broken tool returns an error Observation — the agent should adapt
# WHY: This is the most common production failure mode for agents

import re
import time
import requests


# ============================================================
# Define a deliberately broken tool
# ============================================================
# WHY: Simulates a real-world failure: API down, database unreachable, timeout
# WHY: The agent should detect the error and fall back to another approach
def tool_broken_lookup(query: str) -> str:
    """A deliberately broken tool that always raises an exception."""
    # WHY: This is intentional — we want to see how the agent handles the failure
    # WHY: In production, this could be a flaky API, a down database, or a network timeout
    raise ConnectionError("Database connection failed — server unreachable.")


# ============================================================
# Build a tool registry that includes the broken tool
# ============================================================
# WHY: Create a SEPARATE registry so we do not corrupt the main one
edge_case_tools = dict(TOOLS)  # WHY: Copy existing (working) tools

# WHY: Add the broken tool with a plausible description
# WHY: "database_search" sounds more specific than "rag_lookup", so the LLM may prefer it
edge_case_tools["database_search"] = {
    "description": "Search the company database for employee records. Input: search query.",
    "fn": tool_broken_lookup,
}

# WHY: Print the registry so the learner sees all 5 tools
print("Tools for this demo:")
for name in edge_case_tools:
    # WHY: Mark the broken tool so the learner knows which one will fail
    marker = " [BROKEN]" if name == "database_search" else ""
    print(f"  - {name}{marker}")

# WHY: Choose a question that the broken tool SEEMS relevant for
# WHY: The agent should try database_search, fail, then recover via rag_lookup
question = "What are the office locations for Horizon Tech?"

# WHY: Print expected behavior so the learner can follow along
print(f"\nQuestion: {question}")
print("\nExpected behavior:")
print("  1. Agent may try database_search first (it sounds relevant)")
print("  2. database_search raises ConnectionError -> agent gets error Observation")
print("  3. Agent reasons about the error and falls back to rag_lookup")
print("  4. rag_lookup succeeds -> agent provides Final Answer")
print()

# WHY: Wrap in try/except for Ollama connection errors (separate from tool errors)
try:
    # WHY: Use the production react_agent — it already catches tool exceptions
    # WHY: The react_agent converts tool exceptions into Observation strings
    answer = react_agent(question, edge_case_tools, max_steps=8)
    print(f"\n{'='*50}")
    print(f"FINAL ANSWER: {answer}")
    print(f"{'='*50}")
    # WHY: Explain the takeaway explicitly
    print("\nThe agent recovered from the broken tool and found another path.")
    print("This is why error handling in tools matters — return errors as Observations,")
    print("not as crashes. The agent can reason about errors and adapt.")
except requests.ConnectionError:
    # WHY: Graceful failure if Ollama itself is not running (different from tool error)
    print("Error: Cannot connect to Ollama. Run: ollama serve")
except Exception as e:
    # WHY: Catch-all for unexpected errors outside the agent loop
    print(f"Error: {e}")

---
# Part III: Advanced Reasoning Patterns
*Line cook → sous chef → head chef.*

---

You built a ReAct agent — the line cook who follows orders step by step. Now we climb the kitchen hierarchy. A sous chef plans ahead. A head chef evaluates multiple approaches before committing.

In this section you will build:
- **Tree of Thought** — generate multiple plans, evaluate each, pick the best (the head chef designing menus)
- **Planning Loop** — decompose a complex goal into subtasks, execute each, synthesize results (the sous chef planning dinner service)
- **Deliberate Mistake** — what happens when the agent has no memory between steps (the chef with amnesia)

## 13. The Reasoning Family

Every pattern in this table is a different way for an LLM to reason before (or while) acting. They form a hierarchy of sophistication — and cost:

| Pattern | How It Works | Kitchen Analogy | Best For |
|---|---|---|---|
| **Chain of Thought** | Think step by step, then act | Chef talks through recipe before cooking | Simple multi-step reasoning |
| **ReAct** | Interleave reasoning and action | Chef cooks step by step, tasting as they go | Tool-using tasks |
| **Tree of Thought** | Generate multiple plans, evaluate, pick best | Chef designs 3 menus, picks the winner | High-stakes decisions |
| **Planning Loop** | Decompose goal → plan subtasks → execute → synthesize | Sous chef plans the whole dinner service | Complex multi-part goals |
| **Reflection** | Execute → evaluate → critique → improve | Chef tastes, adjusts seasoning, replates | Quality-critical output |

We built **ReAct** in Part II. **Chain of Thought** was covered in the RAG notebook (Section 25) — it is the foundation these patterns build on. Now we build **Tree of Thought** and **Planning Loops**.

---

### How They Relate

```
Chain of Thought (think, then act once)
    └── ReAct (think, act, observe, repeat)
            ├── Tree of Thought (generate N plans, evaluate, pick best)
            ├── Planning Loop (decompose → execute subtasks → synthesize)
            └── Reflection (execute → critique → improve → repeat)
```

Each pattern adds a layer of sophistication — and a layer of cost. CoT is one LLM call. ReAct is N calls (one per step). Tree of Thought is 3N+ calls (generate + evaluate + select). The question is always: **does the quality gain justify the cost?**

In [ ]:
# ============================================================
# Section 14 — Hello World: Tree of Thought
# ============================================================
# WHY: Tree of Thought (ToT) explores multiple reasoning paths before committing
# WHY: Instead of one plan, generate 3, evaluate each, pick the best
# WHY: This trades compute time for decision quality — essential for high-stakes choices

# WHY: requests needed for Ollama HTTP calls
# WHY: re imported for potential response parsing if needed downstream
import requests
import re

# ============================================================
# Task: Plan a 3-day company offsite for 50 people, $10,000 budget
# ============================================================
# WHY: This task is open-ended enough that multiple valid plans exist
# WHY: Budget constraint forces the LLM to make tradeoffs — perfect for ToT

OFFSITE_TASK = (
    "Plan a 3-day company offsite for 50 people with a $10,000 budget. "
    "Include: daily schedule, venue type, activities, and budget breakdown."
)


def tree_of_thought(task, num_candidates=3):
    """Run a Tree of Thought: Generate, Evaluate, Select.

    WHY: Three distinct LLM calls for three distinct purposes:
      1. Generate = creative (high diversity)
      2. Evaluate = analytical (score each plan)
      3. Select = decisive (pick the winner)

    Args:
        task: The problem or goal to solve.
        num_candidates: How many candidate plans to generate.

    Returns:
        Dict with 'plans', 'evaluations', and 'winner' keys.
    """
    # ==========================================================
    # STEP 1: GENERATE — Ask LLM to produce N candidate plans
    # ==========================================================
    # WHY: We ask for ALL candidates in one call to keep cost low
    # WHY: Numbered format makes parsing easier and more reliable
    # WHY: "Be creative and make each plan DIFFERENT" encourages diversity
    generate_prompt = (
        f"You are an event planner. Generate exactly {num_candidates} DIFFERENT plans for:\n"
        f"{task}\n\n"
        f"For each plan, provide:\n"
        f"- Plan name\n"
        f"- Venue type\n"
        f"- 3-day schedule (brief)\n"
        f"- Key activities\n"
        f"- Budget breakdown\n\n"
        f"Label them Plan 1, Plan 2, Plan 3. Be creative and make each plan DIFFERENT."
    )
    # WHY: ollama_generate for single-turn — no conversation history needed
    print("=" * 60)
    print("STEP 1: GENERATE — Creating 3 candidate plans...")
    print("=" * 60)
    plans_text = ollama_generate(generate_prompt, model="mistral")
    # WHY: Truncate output for readability — plans can be long
    print(plans_text[:1500] + ("..." if len(plans_text) > 1500 else ""))

    # ==========================================================
    # STEP 2: EVALUATE — Ask LLM to score each plan
    # ==========================================================
    # WHY: Separate evaluation call ensures the LLM judges objectively
    # WHY: Scoring on 3 dimensions forces structured comparison
    # WHY: The LLM sees all plans at once for relative comparison
    evaluate_prompt = (
        f"You are an event planning critic. Evaluate these {num_candidates} plans:\n\n"
        f"{plans_text}\n\n"
        f"Score EACH plan on three criteria (1-5 scale):\n"
        f"1. Feasibility — can this realistically be done within the budget?\n"
        f"2. Engagement — will employees actually enjoy this?\n"
        f"3. Budget fit — does the breakdown add up and stay under $10,000?\n\n"
        f"Format:\n"
        f"Plan 1: Feasibility=X, Engagement=X, Budget=X, Total=X\n"
        f"Plan 2: Feasibility=X, Engagement=X, Budget=X, Total=X\n"
        f"Plan 3: Feasibility=X, Engagement=X, Budget=X, Total=X\n"
        f"Best plan: Plan N"
    )
    # WHY: Second LLM call — the evaluator
    print(f"\n{'=' * 60}")
    print("STEP 2: EVALUATE — Scoring each plan...")
    print("=" * 60)
    evaluation_text = ollama_generate(evaluate_prompt, model="mistral")
    print(evaluation_text)

    # ==========================================================
    # STEP 3: SELECT — Pick the highest-scoring plan
    # ==========================================================
    # WHY: Third LLM call asks for a clean summary of the winning plan
    # WHY: This separates "which plan won" from "present it clearly"
    select_prompt = (
        f"Based on this evaluation:\n\n{evaluation_text}\n\n"
        f"State which plan won and give a 3-sentence executive summary of that plan. "
        f"Start with: 'The winning plan is Plan N.'"
    )
    # WHY: Final LLM call — the selector
    print(f"\n{'=' * 60}")
    print("STEP 3: SELECT — Picking the winner...")
    print("=" * 60)
    winner_text = ollama_generate(select_prompt, model="mistral")
    print(winner_text)

    # WHY: Return structured result for downstream use
    # WHY: Keeping all three stages lets the learner inspect each stage independently
    return {
        "plans": plans_text,
        "evaluation": evaluation_text,
        "winner": winner_text,
    }


# --- Run Tree of Thought ---
# WHY: Wrap in try/except — Ollama may not be running
# WHY: The default num_candidates=3 balances diversity vs. cost
try:
    result = tree_of_thought(OFFSITE_TASK)
    # WHY: Print the punchline so the pattern is explicit
    print(f"\n{'=' * 60}")
    print("That's Tree of Thought. Generate \u2192 Evaluate \u2192 Select.")
    # WHY: Explicitly state the cost/benefit tradeoff — the core ToT insight
    print(f"Cost: 3 LLM calls (vs. 1 for a direct answer).")
    print(f"Benefit: The best of 3 ideas, not just the first idea.")
    print(f"{'=' * 60}")
except requests.ConnectionError:
    # WHY: Graceful failure with clear instructions
    print("Error: Cannot connect to Ollama. Run: ollama serve")
except Exception as e:
    # WHY: Catch-all so the notebook does not crash
    print(f"Error (is Ollama running with mistral?): {e}")

## 15. "Why Does This Work?" — Exploration vs. Exploitation

Tree of Thought trades **compute for quality**. Instead of committing to the first idea (exploitation), it explores multiple paths before choosing (exploration). This is the same tradeoff in every search algorithm, from chess engines to Monte Carlo tree search.

> In kitchen terms: a head chef does not cook the first dish that comes to mind. The head chef designs three menus, evaluates each against the guest list and budget, and picks the winner. The extra time spent planning saves time (and reputation) during service.

### When to Use Which Pattern

| Pattern | Cost | Latency | Quality | Use When |
|---|---|---|---|---|
| **CoT** | Low (1 call) | Fast | Good | Simple reasoning — "think through this step by step" |
| **ReAct** | Medium (N calls) | Medium | Good | Tool-using tasks — the agent needs to look things up or calculate |
| **ToT** | High (3N+ calls) | Slow | Best | High-stakes decisions — wrong answer is expensive |
| **Planning** | High (N+1 calls) | Slow | Best | Complex multi-part goals — need to break down and coordinate |

**Rule of thumb:** Use ReAct for routine tasks where speed matters. Use ToT when the cost of a wrong decision exceeds the cost of three extra LLM calls. Use Planning when the goal has 3+ subtasks that need coordination.

---

> **Architect's Note:** Perplexity's Deep Research feature uses planning loops combined with reflection. It decomposes a research question into 10-20 sub-queries, executes each against different sources, synthesizes results, then critiques its own synthesis before presenting the final answer. This produces substantially more thorough results than a single ReAct pass, which typically consults 2-3 sources. The tradeoff: Deep Research takes 30-60 seconds vs. 2-3 seconds for standard Perplexity. The quality gain justifies the cost for complex research questions — but not for "what time is it in Tokyo?"


In [ ]:
# ============================================================
# Section 16 — Planning Loop Agent
# ============================================================
# WHY: A Planning Loop decomposes a complex question into subtasks
# WHY: Each subtask is executed independently, then results are synthesized
# WHY: This is how a sous chef plans dinner service: break down the menu,
#      assign each dish to a station, cook in parallel, plate together

# WHY: requests for Ollama, re for parsing, json for structured output
import requests
import re
import json


def planning_loop_agent(question, tools, max_subtasks=5):
    """Execute a Planning Loop: Decompose, Execute subtasks, Synthesize.

    WHY: Planning Loops outperform ReAct for complex multi-part questions.
    WHY: ReAct reasons step by step — it can lose the big picture after 5+ steps.
    WHY: Planning Loops maintain the big picture by decomposing upfront.

    Args:
        question: The complex question to answer.
        tools: Dict of tool_name -> {description, fn}.
        max_subtasks: Safety limit on decomposition.

    Returns:
        Synthesized final answer as a string.
    """
    # ==========================================================
    # STEP 1: DECOMPOSE — Break the question into subtasks
    # ==========================================================
    # WHY: The planner sees the whole question and breaks it into pieces
    # WHY: Each subtask should be answerable with a single tool call
    # WHY: Numbered list format is easy for the LLM to produce and for us to parse
    tool_names = ", ".join(tools.keys())
    decompose_prompt = (
        f"You are a planning agent. Break this question into 2-{max_subtasks} subtasks:\n\n"
        f"Question: {question}\n\n"
        f"Available tools: {tool_names}\n\n"
        f"List each subtask on its own line, numbered:\n"
        f"1. <subtask description>\n"
        f"2. <subtask description>\n"
        f"...\n\n"
        f"Each subtask should be specific enough to answer with one tool call."
    )

    # WHY: First LLM call — the planner/decomposer
    print("=" * 60)
    print("STEP 1: DECOMPOSE — Breaking question into subtasks...")
    print("=" * 60)
    decomposition = ollama_generate(decompose_prompt, model="mistral")
    print(decomposition)

    # WHY: Parse numbered subtasks from the LLM output
    # WHY: Regex matches lines starting with a digit and period
    subtask_lines = re.findall(r"\d+\.\s*(.+)", decomposition)
    # WHY: Fallback — if parsing fails, treat each line as a subtask
    if not subtask_lines:
        subtask_lines = [line.strip() for line in decomposition.strip().split("\n") if line.strip()]

    # WHY: Safety limit to prevent runaway decomposition
    # WHY: LLMs sometimes over-decompose, creating 10+ subtasks for a 3-part question
    subtask_lines = subtask_lines[:max_subtasks]
    print(f"\nParsed {len(subtask_lines)} subtasks.")

    # ==========================================================
    # STEP 2: EXECUTE — Run each subtask using the ReAct agent
    # ==========================================================
    # WHY: Each subtask gets its own agent run — independent context
    # WHY: This prevents context pollution between unrelated subtasks
    # WHY: Like a sous chef assigning each dish to a different station
    subtask_results = []
    print(f"\n{'=' * 60}")
    print("STEP 2: EXECUTE — Running each subtask...")
    print("=" * 60)

    # WHY: Sequential execution — subtasks run one at a time for simplicity
    # WHY: In production, independent subtasks could run in parallel for speed
    for i, subtask in enumerate(subtask_lines, 1):
        print(f"\n--- Subtask {i}: {subtask} ---")
        # WHY: Use react_agent for each subtask — it handles tool calls
        # WHY: Lower max_steps (4) since each subtask is simple
        # WHY: verbose=False to reduce output noise during multi-subtask runs
        try:
            result = react_agent(subtask, tools, max_steps=4, verbose=False)
        except Exception as e:
            # WHY: Subtask failure should not crash the whole plan
            result = f"Error executing subtask: {e}"
        subtask_results.append({"subtask": subtask, "result": result})
        # WHY: Print each result so the learner can follow progress
        print(f"   Result: {result[:200]}{'...' if len(str(result)) > 200 else ''}")

    # ==========================================================
    # STEP 3: SYNTHESIZE — Combine results into a coherent answer
    # ==========================================================
    # WHY: The synthesizer sees ALL subtask results and combines them
    # WHY: This is the sous chef checking all dishes before plating together
    # WHY: The synthesizer can identify gaps or contradictions
    results_text = "\n".join(
        f"Subtask {i+1} ({r['subtask']}): {r['result']}"
        for i, r in enumerate(subtask_results)
    )
    # WHY: Synthesis prompt asks for a unified answer, not just concatenation
    synthesize_prompt = (
        f"You are a synthesis agent. Combine these subtask results into one clear answer.\n\n"
        f"Original question: {question}\n\n"
        f"Subtask results:\n{results_text}\n\n"
        f"Provide a clear, unified answer that addresses the original question. "
        f"If any subtask returned an error, note what information is missing."
    )

    # WHY: Final LLM call — the synthesizer
    print(f"\n{'=' * 60}")
    print("STEP 3: SYNTHESIZE — Combining results...")
    print("=" * 60)
    synthesis = ollama_generate(synthesize_prompt, model="mistral")
    print(synthesis)

    return synthesis


# ============================================================
# Demo: A question that requires RAG + calculator + synthesis
# ============================================================
# WHY: This question has 3 natural subtasks:
#   1. Look up PTO policy (RAG)
#   2. Look up health insurance (RAG)
#   3. Calculate total dollar value (calculator)
# WHY: A single ReAct agent might lose track after 5+ steps; planning stays organized
PLANNING_QUESTION = (
    "Compare PTO and health insurance at Horizon Tech, "
    "and calculate the total dollar value of these benefits "
    "for a 5-year employee making $85,000 per year."
)

# WHY: Print expected behavior so the learner knows what to look for
print(f"Question: {PLANNING_QUESTION}\n")
print("Expected decomposition:")
print("  1. Look up PTO policy \u2192 find days for 5-year employee")
print("  2. Look up health insurance \u2192 find company contribution")
print("  3. Calculate dollar value of PTO + insurance combined")
print()

# WHY: Wrap in try/except — Ollama may not be running
# WHY: max_subtasks=4 caps decomposition — prevents the planner from over-splitting
try:
    answer = planning_loop_agent(PLANNING_QUESTION, TOOLS, max_subtasks=4)
    # WHY: Summarize the pattern so the learner internalizes it
    print(f"\n{'=' * 60}")
    print("That's a Planning Loop. Decompose \u2192 Execute \u2192 Synthesize.")
    print("The sous chef planned the service before the kitchen started cooking.")
    print(f"{'=' * 60}")
except requests.ConnectionError:
    # WHY: Graceful failure with clear instructions
    print("Error: Cannot connect to Ollama. Run: ollama serve")
except Exception as e:
    # WHY: Catch-all so the notebook does not crash
    print(f"Error (is Ollama running with mistral?): {e}")

## 17. Deliberate Mistake: Agent Without Memory

What happens when you clear the message history between steps?

The agent has **no memory** of what it already did. It re-asks the same questions, re-calls the same tools, and loops until `max_steps`. Every step looks like step 1 — because to the agent, it *is* step 1.

> In kitchen terms: imagine a chef with amnesia. They chop the onions, forget they chopped them, and start chopping again. They heat the broth, forget they heated it, and heat it again. The dish never gets finished because the chef cannot remember what is already done.

This is not a hypothetical — it is one of the most common agent bugs in production. If the message history is accidentally truncated (to save tokens), reset (due to a bug), or not passed between calls (stateless architecture), the agent enters an infinite loop of repeated actions.

**The fix:** Message history must accumulate across steps. Each Thought, Action, and Observation stays in the history so the LLM can see what it already tried.

The next cell demonstrates this side by side.

In [ ]:
# ============================================================
# Section 17 — Memory-Less Agent Demo
# ============================================================
# WHY: Demonstrates why message history (memory) is essential
# WHY: Same question, same tools — only difference is whether memory persists
# WHY: The memory-less agent repeats itself; the normal agent converges

# WHY: requests for Ollama, re for parsing, time for timing
import requests
import re
import time


def memoryless_agent(question, tools, max_steps=5):
    """Run an agent that CLEARS message history between steps.

    WHY: This is the Deliberate Mistake — each step is isolated.
    WHY: The LLM never sees prior steps, so it repeats the same actions.
    WHY: Returns step details so we can compare with the normal agent.

    Args:
        question: The question to answer.
        tools: Tool registry dict.
        max_steps: Max iterations before forced stop.

    Returns:
        Dict with 'steps', 'actions', and 'final_answer' keys.
    """
    # WHY: Build the system prompt from the tool registry
    system = build_react_prompt(tools)
    # WHY: Track what happens at each step for comparison
    actions_taken = []
    final_answer = None

    for step in range(1, max_steps + 1):
        # ==========================================================
        # KEY DIFFERENCE: Fresh messages every step — NO HISTORY
        # ==========================================================
        # WHY: This is the bug. Each step starts from scratch.
        # WHY: The LLM only sees the system prompt and the original question.
        # WHY: It has NO idea what it already tried.
        messages = [
            {"role": "system", "content": system},
            {"role": "user", "content": question}
        ]

        # WHY: Call the LLM — it thinks this is step 1 every time
        response = ollama_chat(messages, model="mistral")
        print(f"  Step {step}: ", end="")

        # WHY: Check for Final Answer (unlikely — it usually tries a tool first)
        if "Final Answer:" in response:
            final_answer = response.split("Final Answer:")[-1].strip()
            print(f"Final Answer: {final_answer[:80]}")
            actions_taken.append("FINAL_ANSWER")
            break

        # WHY: Parse the action to see which tool it tries
        action_match = re.search(r"Action:\s*(\w+)\((.+?)\)", response, re.DOTALL)
        if action_match:
            tool_name = action_match.group(1).strip()
            tool_input = action_match.group(2).strip().strip("\"'")
            actions_taken.append(f"{tool_name}({tool_input[:30]})")
            print(f"Action: {tool_name}({tool_input[:40]})")
            # WHY: Execute the tool — but the result is thrown away
            # WHY: because the history is cleared next step anyway
            if tool_name in tools:
                try:
                    result = tools[tool_name]["fn"](tool_input)
                except Exception:
                    # WHY: Swallow error — the point is that results are lost
                    result = "Error"
                # WHY: Even if we get a result, the next step will not see it
        else:
            # WHY: No parseable action — the LLM is confused
            actions_taken.append("UNPARSEABLE")
            print(f"Unparseable: {response[:60]}...")

    # WHY: Return structured data for comparison
    return {
        "steps": len(actions_taken),
        "actions": actions_taken,
        "final_answer": final_answer,
    }


# ============================================================
# Side-by-side comparison
# ============================================================
# WHY: Same question for both agents — only memory differs
# WHY: PTO question requires RAG lookup then possibly calculator
MEMORY_QUESTION = "How many PTO days does a 5-year employee get at Horizon Tech?"

# WHY: Print setup so the learner knows what to expect
print("=" * 60)
print("QUESTION:", MEMORY_QUESTION)
print("=" * 60)

try:
    # --- Run 1: Memory-less agent ---
    # WHY: This agent clears history between steps — it will repeat itself
    print("\n--- RUN 1: Agent WITHOUT Memory (history cleared each step) ---")
    no_memory = memoryless_agent(MEMORY_QUESTION, TOOLS, max_steps=5)

    # --- Run 2: Normal agent ---
    # WHY: This agent keeps history — it will converge quickly
    print("\n--- RUN 2: Agent WITH Memory (history preserved) ---")
    with_memory_answer = react_agent(MEMORY_QUESTION, TOOLS, max_steps=5, verbose=False)
    print(f"  Final answer: {with_memory_answer}")

    # ============================================================
    # Comparison summary
    # ============================================================
    # WHY: Make the difference explicit and quantifiable
    print(f"\n{'=' * 60}")
    print("COMPARISON")
    print(f"{'=' * 60}")
    print(f"  Without memory: {no_memory['steps']} steps, actions: {no_memory['actions']}")
    # WHY: Check for repeated actions — the hallmark of the memory bug
    unique_actions = set(no_memory["actions"])
    if len(unique_actions) < len(no_memory["actions"]):
        print(f"    ^ REPEATED ACTIONS detected ({len(no_memory['actions'])} total, {len(unique_actions)} unique)")
    print(f"  Without memory answer: {no_memory['final_answer'] or 'None (max steps hit)'}")
    print(f"  With memory answer: {with_memory_answer}")
    # WHY: Print the punchline
    print(f"\n  Memory is what separates an agent from a loop.")
    print(f"  Without memory, every step is step 1.")
except requests.ConnectionError:
    # WHY: Graceful failure if Ollama is not running
    print("Error: Cannot connect to Ollama. Run: ollama serve")
except Exception as e:
    # WHY: Catch-all so the notebook does not crash
    print(f"Error (is Ollama running with mistral?): {e}")

## 18. Questions You Should Be Asking

You have now built three reasoning patterns (ReAct, Tree of Thought, Planning Loop) and seen one critical failure mode (memory loss). These questions should be forming:

1. **When should you use ReAct vs. Tree of Thought vs. Planning Loops?** — ReAct for routine tool use. ToT for high-stakes single decisions. Planning for complex multi-part goals. What about tasks that combine all three?

2. **How do you measure agent efficiency?** — Steps per answer, tokens per answer, latency per answer. A 3-step agent that answers correctly is better than a 7-step agent that also answers correctly. How do you benchmark this?

3. **Tree of Thought generates 3x+ the tokens of ReAct. When is that cost justified?** — When the cost of a wrong decision exceeds the cost of 3 extra LLM calls. How do you quantify "cost of wrong decision" for your use case?

4. **How do you test agents when outputs are non-deterministic?** — Run the same question 10 times. If the agent answers correctly 8/10 times, is that acceptable? What about 6/10? How do you set a reliability threshold?

5. **What happens when the planner's decomposition is wrong?** — The planner breaks "compare X and Y" into subtasks, but misses a subtask. The synthesis is incomplete. How do you detect this? Can the synthesizer identify gaps?

6. **How do you detect when the agent is hallucinating vs. reasoning?** — The LLM outputs "Thought: I know that PTO (Paid Time Off) is 20 days" without calling rag_lookup. It may be right (from training data) or wrong (hallucination). How do you enforce tool use for factual claims?

7. **What is acceptable latency for an agent response?** — Interactive (user waiting): 5-10 seconds. Batch (background processing): minutes. How do you set expectations and fallback to simpler patterns when latency exceeds the threshold?

8. **Can you mix patterns?** — ReAct for subtasks within a Planning Loop. ToT for critical subtasks. Reflection after synthesis. The patterns are composable — but complexity compounds. When is mixing worth it?

9. **How do you choose max_steps?** — Too low and the agent truncates answers. Too high and it wastes tokens wandering. Is there a formula, or is it empirical tuning per task type?

10. **Should temperature be different for different stages?** — Routing and tool selection need precision (temperature 0.0). Creative tasks like brainstorming benefit from diversity (temperature 0.7). Can you vary temperature within a single agent run?

---
# Part IV: Multi-Agent Systems & MCP
*The Kitchen Brigade.*

---

A single chef can cook a meal. But a restaurant needs a **kitchen brigade** — specialists who each do one thing exceptionally well, coordinated by a head chef. A pastry chef does not season the steak. A saucier does not bake bread. Specialization is what separates a home kitchen from a Michelin-star restaurant.

This section teaches you to build multi-agent systems — specialized agents that collaborate on complex tasks — and to standardize how agents discover and use tools via the Model Context Protocol (MCP).

**What you will build:**
1. A **multi-agent router** — classify the question, dispatch to the right specialist
2. A **router with fallback** — graceful handling when no specialist matches
3. An **MCP server and client** — the emerging standard for agent-tool communication
4. An **MCP-powered agent** — tools discovered at runtime, not hardcoded

## 19. Why Multi-Agent?

A single agent with 10+ tools starts to break down. The failure modes are predictable:

- **Prompt complexity:** The system prompt grows with every tool. At 10+ tools, the LLM struggles to pick the right one — like a chef staring at 50 knives trying to remember which one is for bread.
- **Tool conflicts:** A "search" tool and a "lookup" tool sound similar. The LLM confuses them and picks wrong.
- **Context overflow:** Long multi-step conversations exhaust the context window. Earlier steps fall off.
- **Specialization:** An HR question needs different prompting than a math question. One prompt cannot optimize for both.

The solution: **multiple specialized agents**, each with a focused prompt and a small set of tools, coordinated by a router or orchestrator.

### Multi-Agent Patterns

| Pattern | Description | Kitchen Analogy | When |
|---|---|---|---|
| **Single Agent** | One LLM, all tools | Solo chef cooking everything | < 5 tools, simple tasks |
| **Router** | Classifier → specialist agents | Maître d' sends orders to the right station | Distinct task categories |
| **Orchestrator** | Head agent decomposes → delegates → synthesizes | Head chef directs the entire brigade | Complex, multi-part goals |
| **Pipeline** | Agent A output → Agent B input → Agent C input | Assembly line: prep → cook → plate | Sequential processing |

---

> **Architect's Note:** Atlassian's Rovo uses a hierarchical routing pattern. A top-level orchestrator classifies the user's intent (Jira task, Confluence search, Bitbucket code review, general question) and delegates to specialized subagents. Each subagent has its own prompt, tools, and memory — the Jira agent speaks "project management," the Confluence agent speaks "knowledge base." This separation of concerns mirrors the kitchen brigade: the pastry chef does not season the steak. (Source: [Atlassian Engineering Blog — Building Rovo Agents](https://www.atlassian.com/engineering/building-rovo-agents))

In [ ]:
# ============================================================
# Section 20 — Hello World: Multi-Agent Router
# ============================================================
# WHY: A router agent classifies the question and dispatches to the right specialist
# WHY: Each specialist has ONLY its relevant tools — smaller decision space
# WHY: Like a maître d' who sends the steak order to the grill station, not pastry

# WHY: requests for Ollama, re for parsing, time for latency tracking
import requests
import re
import time


# ============================================================
# Define specialist tool registries — each station gets its own equipment
# ============================================================
# WHY: Separate registries enforce tool isolation per specialist

# WHY: HR specialist gets only RAG-related tools
# WHY: Smaller tool set = fewer wrong choices for the LLM
HR_TOOLS = {
    "rag_lookup": TOOLS["rag_lookup"],
    "get_datetime": TOOLS["get_datetime"],
}

# WHY: Math specialist gets only calculation-related tools
# WHY: No RAG distraction — this agent focuses purely on math
MATH_TOOLS = {
    "calculator": TOOLS["calculator"],
}

# WHY: classify_question is the "maître d'" — it reads the order and picks the station
def classify_question(question):
    """Classify a question as 'hr', 'math', or 'both'.

    WHY: The router uses a lightweight LLM call just for classification.
    WHY: Classification is cheaper than running the wrong specialist.
    WHY: Returns a single word so parsing is trivial.
    """
    # WHY: Very constrained prompt — we only want one word back
    # WHY: Examples improve classification accuracy (few-shot prompting)
    classify_prompt = (
        "Classify this question into EXACTLY ONE category.\n"
        "Categories:\n"
        "  hr — questions about company policies, PTO, insurance, benefits, offices\n"
        "  math — questions about calculations, percentages, dollar amounts\n"
        "  both — questions that need BOTH company data AND calculations\n\n"
        "Examples:\n"
        "  'What is the PTO policy?' \u2192 hr\n"
        "  'What is 15% of $85,000?' \u2192 math\n"
        "  'How many PTO days does a 5-year employee get and what is the dollar value?' \u2192 both\n\n"
        f"Question: {question}\n\n"
        "Respond with ONLY the category (hr, math, or both). Nothing else."
    )
    # WHY: Single LLM call for classification — fast and cheap
    result = ollama_generate(classify_prompt, model="mistral")
    # WHY: Extract just the category word — LLM may add extra text
    category = result.strip().lower().split()[0] if result.strip() else "hr"
    # WHY: Normalize to known categories
    if category not in ("hr", "math", "both"):
        # WHY: Default to 'hr' if classification is unclear — safer than guessing math
        category = "hr"
    return category


def multi_agent_router(question):
    """Route a question to the appropriate specialist agent.

    WHY: Classify, Dispatch, Return. Three simple steps.
    WHY: Each specialist runs independently with its own focused tools.
    WHY: For 'both' questions, run HR first (get facts), then Math (calculate).

    Args:
        question: The user's question.

    Returns:
        The specialist agent's answer.
    """
    # WHY: Step 1 — Classify the question
    print("--- ROUTER: Classifying question... ---")
    category = classify_question(question)
    print(f"--- ROUTER: Category = '{category}' ---\n")

    # WHY: Step 2 — Dispatch to the right specialist
    if category == "hr":
        # WHY: HR specialist has RAG + datetime tools
        print("--- DISPATCHING to HR Specialist ---")
        return react_agent(question, HR_TOOLS, max_steps=5, verbose=True)

    elif category == "math":
        # WHY: Math specialist has only the calculator tool
        print("--- DISPATCHING to Math Specialist ---")
        return react_agent(question, MATH_TOOLS, max_steps=5, verbose=True)

    elif category == "both":
        # WHY: For mixed questions, run both specialists and combine
        # WHY: HR first to get the facts, then Math to calculate
        print("--- DISPATCHING to HR Specialist (gather facts) ---")
        hr_answer = react_agent(question, HR_TOOLS, max_steps=5, verbose=False)
        print(f"  HR result: {hr_answer[:150]}")

        # WHY: Feed HR facts into the math question for calculation
        print("\n--- DISPATCHING to Math Specialist (calculate) ---")
        math_question = f"Based on this information: {hr_answer}\n\nNow answer: {question}"
        math_answer = react_agent(math_question, MATH_TOOLS, max_steps=5, verbose=False)
        print(f"  Math result: {math_answer[:150]}")

        # WHY: Combine both results into a coherent answer
        return f"HR Info: {hr_answer}\nCalculation: {math_answer}"

    # WHY: Fallback (should not reach here, but defensive coding)
    return react_agent(question, TOOLS, max_steps=5, verbose=True)


# ============================================================
# Demo: Three different question types
# ============================================================
# WHY: Each question exercises a different routing path
# WHY: Shows the router correctly classifying and dispatching each type

demo_questions = [
    # WHY: Pure HR question — should route to HR specialist
    ("What is the PTO policy at Horizon Tech?", "hr"),
    # WHY: Pure math question — should route to Math specialist
    ("What is 15% of $85,000?", "math"),
    # WHY: Mixed question — should use both specialists
    ("How many PTO days does a 5-year employee get, and what's the dollar value at $85,000/year?", "both"),
]

# WHY: Three demos cover all routing paths: hr-only, math-only, and both-combined
# WHY: Comparing expected vs. actual classification validates the router's accuracy
try:
    for question, expected in demo_questions:
        print(f"\n{'=' * 60}")
        print(f"QUESTION: {question}")
        # WHY: Printing expected route lets the learner verify the classifier works
        print(f"EXPECTED ROUTE: {expected}")
        print(f"{'=' * 60}")
        answer = multi_agent_router(question)
        print(f"\nFINAL ANSWER: {answer[:300]}")
        print()

    # WHY: Print the punchline
    print(f"{'=' * 60}")
    print("That's multi-agent routing. Classify \u2192 Dispatch \u2192 Specialize.")
    print("Each specialist has fewer tools \u2192 fewer mistakes \u2192 better answers.")
    print(f"{'=' * 60}")

except requests.ConnectionError:
    # WHY: Graceful failure if Ollama is not running
    print("Error: Cannot connect to Ollama. Run: ollama serve")
except Exception as e:
    # WHY: Catch-all so the notebook does not crash
    print(f"Error (is Ollama running with mistral?): {e}")

## 21. "Why Does This Work?" — Specialization Reduces Decision Space

A 10-tool agent must choose from 10 options at every step. A router + 2 specialists (5 tools each) reduces the decision space by half after routing. **Fewer choices = fewer mistakes.**

This is the same principle behind microservices: small, focused services outperform monoliths for complex systems. Each service (agent) has a single responsibility, a clear interface, and independent scaling.

### The Math

| Architecture | Tools per Decision | Chance of Wrong Tool (est.) |
|---|---|---|
| Single agent, 10 tools | 10 | ~30-40% per step |
| Router + 2 specialists, 5 each | 5 (after routing) | ~15-20% per step |
| Router + 4 specialists, 2-3 each | 2-3 (after routing) | ~5-10% per step |

The router adds one extra LLM call (classification), but it saves multiple wasted steps from wrong tool selection. Net result: fewer total tokens, faster answers, higher accuracy.

### When Single Agent Still Wins

Not every system needs multi-agent. Single agents are better when:
- Tool count is low (< 5)
- Tools are clearly distinct (calculator vs. calendar vs. email)
- Latency matters more than accuracy (routing adds one round trip)
- The task is always the same category (no routing needed)

---

> **Architect's Note:** Databricks Genie uses a 4-agent pipeline for text-to-SQL (Structured Query Language): (1) Intent Classifier determines if the query needs data, (2) Schema Agent selects relevant tables and columns, (3) SQL Generator writes the query, (4) Validator checks SQL syntax and safety. Each agent sees only what it needs — the SQL Generator never sees irrelevant tables, reducing hallucination. This pipeline architecture processes millions of queries per day. (Source: [Databricks Engineering Blog — Building Genie](https://www.databricks.com/blog/building-genie))

## 22. Deliberate Mistake: Router Without Fallback

What happens when the router receives a question it was not trained for?

"What's the weather in Tokyo?" — The router only knows "hr" and "math." Without a fallback category, it force-classifies as one of the two. The HR specialist tries to look up weather in the employee handbook. The math specialist tries to calculate weather. Either way: **nonsense**.

> In kitchen terms: a customer orders sushi at a French restaurant. Without a "we don't serve that" option, the kitchen tries to make sushi with French ingredients. The result is not sushi. It is not French food either. It is confusion on a plate.

Every router needs a **fallback** — a category that means "none of the above." The fallback should:
1. Acknowledge what the system *can* do
2. Decline what it *cannot* do
3. Suggest where to go instead

The next cell demonstrates this side by side.

In [ ]:
# ============================================================
# Section 22 — Router Fallback Demo
# ============================================================
# WHY: Demonstrates what happens when a router has no fallback category
# WHY: Same question, two routers — one without fallback, one with
# WHY: The fallback version gracefully declines instead of producing nonsense

# WHY: requests for Ollama connection
import requests


def classify_without_fallback(question):
    """Classify into ONLY 'hr' or 'math' — no escape hatch.

    WHY: This is the broken version. The LLM MUST pick one of two categories.
    WHY: For off-topic questions, it will force-fit into the wrong category.
    """
    # WHY: Only two options — no fallback, no "other", no "unknown"
    prompt = (
        "Classify this question as either 'hr' or 'math'. "
        "You MUST choose one. Respond with only the category.\n\n"
        f"Question: {question}\n\nCategory:"
    )
    # WHY: LLM is forced to pick hr or math even for weather questions
    result = ollama_generate(prompt, model="mistral").strip().lower().split()[0]
    # WHY: Normalize — if LLM outputs something weird, default to hr
    return result if result in ("hr", "math") else "hr"


def classify_with_fallback(question):
    """Classify into 'hr', 'math', or 'unknown' — with escape hatch.

    WHY: The 'unknown' category prevents force-fitting off-topic questions.
    WHY: Better to say 'I cannot help with that' than to give a wrong answer.
    """
    # WHY: Three options including 'unknown' — the safe fallback
    # WHY: Examples help the LLM calibrate what counts as 'unknown'
    prompt = (
        "Classify this question into EXACTLY ONE category.\n"
        "Categories:\n"
        "  hr — company policies, PTO, insurance, benefits, offices\n"
        "  math — calculations, percentages, dollar amounts\n"
        "  unknown — anything that does NOT fit hr or math\n\n"
        "Examples:\n"
        "  'What is the PTO policy?' \u2192 hr\n"
        "  'What is 15% of 200?' \u2192 math\n"
        "  'What is the weather in Tokyo?' \u2192 unknown\n"
        "  'Write me a poem about cats' \u2192 unknown\n\n"
        f"Question: {question}\n\nCategory:"
    )
    # WHY: LLM now has an escape route for off-topic questions
    result = ollama_generate(prompt, model="mistral").strip().lower().split()[0]
    # WHY: Default to 'unknown' (safe) rather than guessing a category
    return result if result in ("hr", "math", "unknown") else "unknown"


# ============================================================
# Test with an off-topic question
# ============================================================
# WHY: Weather is clearly outside HR and math — reveals the fallback gap
OFF_TOPIC = "What's the weather like in Tokyo right now?"

print("=" * 60)
print(f"QUESTION: {OFF_TOPIC}")
print("=" * 60)

try:
    # --- Run 1: No fallback ---
    # WHY: Router is forced to pick hr or math for a weather question
    print("\n--- RUN 1: Router WITHOUT Fallback ---")
    category_1 = classify_without_fallback(OFF_TOPIC)
    print(f"  Classification: '{category_1}'")
    # WHY: Run the (wrong) specialist to show the nonsensical result
    if category_1 == "hr":
        answer_1 = react_agent(OFF_TOPIC, HR_TOOLS, max_steps=3, verbose=False)
    else:
        answer_1 = react_agent(OFF_TOPIC, MATH_TOOLS, max_steps=3, verbose=False)
    # WHY: Print the answer — it will be nonsense or irrelevant
    print(f"  Answer: {answer_1[:200]}")

    # --- Run 2: With fallback ---
    # WHY: Router can now say "unknown" and decline gracefully
    print("\n--- RUN 2: Router WITH Fallback ---")
    category_2 = classify_with_fallback(OFF_TOPIC)
    print(f"  Classification: '{category_2}'")
    if category_2 == "unknown":
        # WHY: Graceful decline — tell the user what you CAN do
        answer_2 = (
            "I can help with HR questions (PTO, insurance, company policies) "
            "and math calculations. For weather information, try a weather service "
            "or ask: 'What are the Horizon Tech office locations?'"
        )
    elif category_2 == "hr":
        # WHY: If it still classifies as hr, run the specialist
        answer_2 = react_agent(OFF_TOPIC, HR_TOOLS, max_steps=3, verbose=False)
    else:
        # WHY: If it classifies as math, run the math specialist
        answer_2 = react_agent(OFF_TOPIC, MATH_TOOLS, max_steps=3, verbose=False)
    # WHY: Print the graceful decline
    print(f"  Answer: {answer_2[:200]}")

    # ============================================================
    # Comparison
    # ============================================================
    # WHY: Make the difference explicit
    print(f"\n{'=' * 60}")
    print("COMPARISON")
    print(f"{'=' * 60}")
    print(f"  Without fallback: classified as '{category_1}' \u2192 nonsensical or irrelevant answer")
    print(f"  With fallback:    classified as '{category_2}' \u2192 graceful decline with guidance")
    # WHY: Print the punchline
    print(f"\n  Always design for the questions your agent CAN'T answer.")
    print(f"  A confident wrong answer is worse than an honest 'I don't know.'")

except requests.ConnectionError:
    # WHY: Graceful failure if Ollama is not running
    print("Error: Cannot connect to Ollama. Run: ollama serve")
except Exception as e:
    # WHY: Catch-all so the notebook does not crash
    print(f"Error (is Ollama running with mistral?): {e}")

## 23. LangGraph — When to Use a Framework

You built agents from scratch. Now you understand what happens inside the black box. Frameworks like LangGraph, CrewAI, and AutoGen automate the plumbing — but the architecture decisions remain yours.

### Framework Comparison

| | Pure Python | LangGraph | CrewAI | AutoGen |
|---|---|---|---|---|
| **Learning curve** | Low | Medium | Medium | High |
| **Control** | Total | High | Medium | Low |
| **Visualization** | DIY | Built-in graph viz | Limited | Limited |
| **Production use** | Manual infra | Checkpointing, streaming, human-in-the-loop | Task orchestration, role-based teams | Multi-agent research, autonomous |
| **State management** | You build it | Built-in state graph | Implicit | Implicit |
| **Best for** | Learning, simple agents | Production single-agent or multi-agent | Role-based team collaboration | Autonomous research agents |

### What Frameworks Add

1. **State management** — Track agent state across steps without manual message lists
2. **Checkpointing** — Save and resume mid-run (critical for long-running agents)
3. **Streaming** — Token-by-token output to the user (feels faster)
4. **Human-in-the-loop** — Pause and ask for human approval before critical actions
5. **Visualization** — See the agent's decision graph (LangGraph generates Mermaid diagrams)
6. **Error recovery** — Automatic retry with backoff on failures

### When to Use What

- **Learning:** Pure Python. You need to understand the loop before abstracting it.
- **Prototyping:** Pure Python or LangGraph. Fast iteration, no framework lock-in.
- **Production (single agent):** LangGraph. Checkpointing and streaming are table stakes.
- **Production (multi-agent):** LangGraph or CrewAI. Orchestration is complex enough to warrant framework support.
- **Research:** AutoGen. Its autonomous agent patterns are designed for open-ended exploration.

> Use frameworks when you need production features. Use pure Python when you need to understand what is happening. You have done the second — the next cell shows the framework equivalent.

In [ ]:
# ============================================================
# Section 23 — LangGraph Equivalent of Our ReAct Agent
# ============================================================
# WHY: Show that the SAME agent we built from scratch can be expressed in LangGraph
# WHY: The trace should be identical — same Thought/Action/Observation loop
# WHY: The difference is infrastructure: checkpointing, streaming, visualization

# WHY: LangGraph may not be installed — wrap everything in try/except
try:
    from langgraph.graph import StateGraph, END
    from typing import TypedDict
    # WHY: Flag tracks whether we can run the full demo
    LANGGRAPH_AVAILABLE = True
    print("\u2713 langgraph imported successfully.")
except ImportError:
    # WHY: Graceful degradation — show the code but do not run it
    LANGGRAPH_AVAILABLE = False
    print("\u2717 langgraph not available. Showing code structure only.")
    print("  Install with: pip install langgraph")


# ============================================================
# LangGraph state definition and nodes
# ============================================================
# WHY: LangGraph uses typed state dicts instead of raw message lists
# WHY: This gives you compile-time validation of your agent's data flow
# WHY: The 'messages' key replaces our manual messages list

if LANGGRAPH_AVAILABLE:
    import re
    import requests

    # WHY: TypedDict defines the shape of state flowing through the graph
    class AgentState(TypedDict):
        # WHY: 'messages' accumulates like our manual list
        messages: list
        # WHY: 'next_action' controls which node runs next
        next_action: str

    # ==========================================================
    # Node: Think — LLM generates Thought + Action or Final Answer
    # ==========================================================
    def think_node(state):
        """LLM generates a Thought + Action or Final Answer.

        WHY: This is the SAME logic as our react_agent's inner loop.
        WHY: In LangGraph, it is a named node instead of inline code.
        """
        # WHY: Build system prompt from global TOOLS — same as our version
        system = build_react_prompt(TOOLS)
        # WHY: Prepend system message if not present
        msgs = state["messages"]
        if not msgs or msgs[0].get("role") != "system":
            msgs = [{"role": "system", "content": system}] + msgs
        # WHY: Call Ollama — identical to our from-scratch agent
        response = ollama_chat(msgs, model="mistral")
        # WHY: Append response to state messages
        new_msgs = msgs + [{"role": "assistant", "content": response}]
        # WHY: Decide next node based on response content
        if "Final Answer:" in response:
            return {"messages": new_msgs, "next_action": "done"}
        elif re.search(r"Action:\s*(\w+)\(", response):
            return {"messages": new_msgs, "next_action": "act"}
        else:
            return {"messages": new_msgs, "next_action": "think"}

    # ==========================================================
    # Node: Act — Parse and execute the tool call
    # ==========================================================
    def act_node(state):
        """Parse and execute the tool call, return Observation.

        WHY: Same dispatch logic as our react_agent — parse Action, call tool.
        WHY: In LangGraph, tool execution is a separate node for clarity.
        """
        # WHY: Get the last assistant message (the one with the Action)
        last_msg = state["messages"][-1]["content"]
        # WHY: Parse Action — same regex as our from-scratch version
        match = re.search(r"Action:\s*(\w+)\((.+?)\)", last_msg, re.DOTALL)
        if match:
            tool_name = match.group(1).strip()
            tool_input = match.group(2).strip().strip("\"'")
            # WHY: Execute tool or return error — identical to our version
            if tool_name in TOOLS:
                try:
                    result = TOOLS[tool_name]["fn"](tool_input)
                    obs = f"Observation: {result}"
                except Exception as e:
                    obs = f"Observation: Tool error \u2014 {e}"
            else:
                obs = f"Observation: Tool '{tool_name}' not found."
        else:
            obs = "Observation: Could not parse action."
        # WHY: Feed observation back as user message — same pattern
        new_msgs = state["messages"] + [{"role": "user", "content": obs}]
        return {"messages": new_msgs, "next_action": "think"}

    def route_next(state):
        """Route to the next node based on state.

        WHY: LangGraph uses conditional edges — this function decides routing.
        WHY: Maps our if/else logic into a graph edge decision.
        """
        # WHY: 'done' ends the graph, 'act' executes a tool, 'think' re-reasons
        if state["next_action"] == "done":
            return END
        elif state["next_action"] == "act":
            return "act"
        else:
            return "think"

    # ==========================================================
    # Build the graph — same agent, graph representation
    # ==========================================================
    # WHY: StateGraph is the LangGraph equivalent of our while loop
    # WHY: Nodes = functions, Edges = control flow
    graph = StateGraph(AgentState)
    # WHY: Add nodes — each node is a step in our agent loop
    graph.add_node("think", think_node)
    graph.add_node("act", act_node)
    # WHY: Set entry point — agent always starts by thinking
    graph.set_entry_point("think")
    # WHY: Conditional edges replace our if/else routing
    graph.add_conditional_edges("think", route_next)
    graph.add_conditional_edges("act", route_next)
    # WHY: Compile the graph — validates edges and produces a runnable
    app = graph.compile()

    # WHY: Print graph structure so the learner sees the topology
    print("\nLangGraph agent compiled successfully.")
    print("Nodes: think \u2192 act \u2192 think \u2192 ... \u2192 END")
    print("Same agent. Same logic. Graph representation.\n")

    # ==========================================================
    # Run the LangGraph agent
    # ==========================================================
    # WHY: Same question as our from-scratch demo for direct comparison
    lg_question = "How many PTO days does a new employee get at Horizon Tech?"
    print(f"Question: {lg_question}")
    print("-" * 40)

    try:
        # WHY: invoke() runs the graph to completion
        # WHY: Initial state has just the user message
        result = app.invoke({
            "messages": [{"role": "user", "content": lg_question}],
            "next_action": "think",
        })
        # WHY: Extract final answer from last assistant message
        for msg in reversed(result["messages"]):
            if msg.get("role") == "assistant" and "Final Answer:" in msg.get("content", ""):
                final = msg["content"].split("Final Answer:")[-1].strip()
                print(f"\nLangGraph Final Answer: {final}")
                break
        else:
            print("\nNo final answer found in LangGraph run.")

        # WHY: Count steps for comparison with our from-scratch version
        assistant_msgs = [m for m in result["messages"] if m.get("role") == "assistant"]
        print(f"Steps: {len(assistant_msgs)}")
        print("\nSame agent. Same trace. LangGraph adds: checkpointing, streaming, viz.")

    except requests.ConnectionError:
        # WHY: Ollama not running
        print("Error: Cannot connect to Ollama. Run: ollama serve")
    except Exception as e:
        # WHY: Catch-all — LangGraph may have version-specific issues
        print(f"LangGraph run error: {e}")
        print("The code structure above is correct \u2014 check langgraph version compatibility.")

else:
    # WHY: If LangGraph is not installed, show what the code WOULD look like
    print("\n--- LangGraph Code Structure (not runnable without install) ---")
    print("  # State definition:")
    print("  class AgentState(TypedDict):")
    print("      messages: list")
    print("      next_action: str")
    print()
    print("  # Graph nodes:")
    print("  graph.add_node('think', think_node)   # LLM reasoning")
    print("  graph.add_node('act', act_node)       # Tool execution")
    print()
    print("  # Edges:")
    print("  graph.set_entry_point('think')")
    print("  graph.add_conditional_edges('think', route_next)")
    print("  graph.add_conditional_edges('act', route_next)")
    print()
    print("  # Run:")
    print("  app = graph.compile()")
    print("  result = app.invoke(initial_state)")
    print()
    print("  Install langgraph to run this: pip install langgraph")

## 24. MCP — The Standardized Kitchen Layout

The **Model Context Protocol (MCP)** is USB for AI tools. Before USB, every device needed its own cable — printers, keyboards, cameras each had a proprietary connector. Before MCP, every LLM framework had its own tool format — LangChain tools, OpenAI function calling, Anthropic tool use, each with different schemas.

MCP standardizes how AI applications discover and call tools, regardless of framework.

| | Without MCP | With MCP |
|---|---|---|
| **Tool format** | Framework-specific (LangChain, OpenAI, etc.) | Standardized JSON (JavaScript Object Notation)-RPC |
| **Discovery** | Hardcoded tool list in the prompt | Dynamic discovery at runtime |
| **Reusability** | Copy-paste tool code per framework | Write once, use everywhere |
| **Kitchen analogy** | Every restaurant has different equipment layout | NSF kitchen code — any chef walks into any kitchen and knows where everything is |

### How MCP Works

```
┌──────────────────┐         ┌──────────────────┐
│   MCP Client     │         │   MCP Server     │
│  (your agent)    │◄───────►│  (tool provider) │
│                  │ JSON-RPC│                  │
│  list_tools()    │───────►│  returns schemas │
│  call_tool()     │───────►│  executes + returns│
└──────────────────┘         └──────────────────┘
```

1. **Client** calls `list_tools()` — discovers what tools are available
2. **Client** calls `call_tool(name, args)` — server executes and returns result
3. **Neither side** knows the other's internals — clean separation

### Transport Options

| Transport | How It Works | Use When |
|---|---|---|
| **stdio** | Process-to-process via stdin/stdout | Local tools (file system, database) |
| **HTTP (HyperText Transfer Protocol)/SSE** | Network request + server-sent events | Remote tools (APIs, cloud services) |

---

> **Architect's Note:** MCP was open-sourced by Anthropic in late 2024 and adopted rapidly: Cursor, Windsurf, GitHub Copilot, Zed, and Claude Desktop integrated MCP servers within months. The protocol uses JSON-RPC over stdio (local) or HTTP/SSE (remote). The key insight: tool providers (database, calendar, Slack) publish MCP servers; AI applications connect as MCP clients. Neither side needs to know about the other's internals. As of early 2026, there are 1,000+ community MCP servers on GitHub covering databases, APIs, file systems, and cloud services.

In [ ]:
# ============================================================
# Section 24 — MCP Server + Client (Pattern Demo)
# ============================================================
# WHY: Demonstrates the MCP pattern: Discover, Call, Result
# WHY: Implements the core MCP concepts manually so you understand the protocol
# WHY: Real MCP uses JSON-RPC over stdio/HTTP — this captures the same logic in Python

# WHY: json for structured tool schemas, typing for type hints
import json
from typing import Dict, Any, List


# ============================================================
# MCP Server — The tool provider (kitchen equipment supplier)
# ============================================================
# WHY: An MCP server exposes tools via two operations:
#   1. list_tools() — returns metadata (name, description, input schema)
#   2. call_tool(name, arguments) — dispatches and returns result
# WHY: The server knows NOTHING about the client — clean separation

class MCPServer:
    """A simplified MCP server that registers and exposes tools.

    WHY: In real MCP, this communicates over stdio or HTTP/SSE.
    WHY: Here we implement the same interface as direct Python calls.
    WHY: The pattern is identical — only the transport layer differs.
    """

    def __init__(self, name):
        # WHY: Server has a name for identification in multi-server setups
        self.name = name
        # WHY: Internal tool registry — same concept as our TOOLS dict
        self._tools = {}

    def register_tool(self, name, description, input_schema, fn):
        """Register a tool with full MCP-compatible metadata.

        WHY: MCP requires input_schema (JSON Schema) so clients know
             what arguments the tool expects — without reading source code.
        WHY: This is the key difference from our simple TOOLS dict:
             MCP tools are self-describing.
        """
        # WHY: Store everything the client needs to discover and call the tool
        self._tools[name] = {
            "name": name,
            "description": description,
            "inputSchema": input_schema,
            "fn": fn,
        }

    def list_tools(self):
        """Return metadata for all registered tools (MCP tools/list).

        WHY: This is the MCP discovery endpoint.
        WHY: Returns ONLY metadata — no function references.
        WHY: The client uses this to build its tool catalog.
        """
        # WHY: Strip the 'fn' key — clients should not see implementation details
        return [
            {
                "name": t["name"],
                "description": t["description"],
                "inputSchema": t["inputSchema"],
            }
            for t in self._tools.values()
        ]

    def call_tool(self, name, arguments):
        """Execute a tool by name with given arguments (MCP tools/call).

        WHY: This is the MCP execution endpoint.
        WHY: Returns a structured result dict (not raw string).
        WHY: Error handling returns error in the result, not as an exception.
        """
        # WHY: Check if tool exists — return error if not
        if name not in self._tools:
            return {"error": f"Tool '{name}' not found on server '{self.name}'."}
        try:
            # WHY: Call the tool function with the provided arguments
            result = self._tools[name]["fn"](**arguments)
            # WHY: Wrap result in MCP-standard response format
            return {"content": [{"type": "text", "text": str(result)}]}
        except Exception as e:
            # WHY: Tool errors become structured responses, not crashes
            return {"error": f"Tool execution error: {e}"}


# ============================================================
# MCP Client — The tool consumer (the chef using equipment)
# ============================================================
# WHY: An MCP client connects to servers, discovers tools, and calls them
# WHY: The client knows NOTHING about tool implementations — only metadata

class MCPClient:
    """A simplified MCP client that discovers and calls tools from servers.

    WHY: In real MCP, this connects over stdio or HTTP/SSE.
    WHY: Here we call server methods directly — same logic, simpler transport.
    """

    def __init__(self):
        # WHY: Client can connect to multiple servers
        self._servers = {}
        # WHY: Unified tool catalog built from all connected servers
        self._tool_catalog = {}

    def connect(self, server):
        """Connect to an MCP server and discover its tools.

        WHY: On connect, the client calls list_tools() to build its catalog.
        WHY: This is dynamic discovery — no hardcoded tool knowledge.
        """
        # WHY: Store server reference for later tool calls
        self._servers[server.name] = server
        # WHY: Discover tools from this server
        tools = server.list_tools()
        # WHY: Add each tool to the unified catalog with server reference
        for tool in tools:
            self._tool_catalog[tool["name"]] = {
                **tool,
                "server": server.name,
            }
        # WHY: Print discovery results so the learner sees what was found
        print(f"Connected to server '{server.name}': discovered {len(tools)} tool(s)")

    def list_all_tools(self):
        """Return all discovered tools across all servers.

        WHY: Unified view of tools from all connected servers.
        """
        return list(self._tool_catalog.values())

    def call_tool(self, name, arguments):
        """Call a tool by name — routes to the correct server automatically.

        WHY: The client does not need to know which server owns the tool.
        WHY: Routing is based on the discovery catalog.
        """
        # WHY: Look up which server owns this tool
        if name not in self._tool_catalog:
            return {"error": f"Tool '{name}' not found in any connected server."}
        # WHY: Get the server and delegate the call
        server_name = self._tool_catalog[name]["server"]
        server = self._servers[server_name]
        return server.call_tool(name, arguments)


# ============================================================
# Demo: Build a server, register a tool, connect a client
# ============================================================

# WHY: Create a math tools server — like a kitchen equipment supplier
math_server = MCPServer("math-tools")
# WHY: Register calculator with a proper JSON Schema for input
math_server.register_tool(
    name="calculator",
    description="Evaluate a math expression safely",
    input_schema={
        "type": "object",
        "properties": {
            "expression": {"type": "string", "description": "Math expression to evaluate"}
        },
        "required": ["expression"],
    },
    # WHY: Same safe calculator as before — restricted eval
    fn=lambda expression: str(eval(expression, {"__builtins__": {}})),
)

# WHY: Create the client and connect to the server
client = MCPClient()
client.connect(math_server)

# WHY: Show the discovery results — what the client learned
print("\nDiscovered tools:")
for tool in client.list_all_tools():
    # WHY: Print the metadata the client received — no implementation details
    print(f"  {tool['name']}: {tool['description']}")
    print(f"    Input schema: {json.dumps(tool['inputSchema'], indent=2)}")

# WHY: Call the tool via the client — the client routes to the right server
print("\n--- Calling calculator(25 * 47) via MCP ---")
result = client.call_tool("calculator", {"expression": "25 * 47"})
print(f"Result: {result}")

# WHY: Print the punchline — what MCP achieves
print(f"\n{'=' * 60}")
print("That's MCP. Discover \u2192 Call \u2192 Result.")
print("No hardcoded tool knowledge needed.")
print("The client learned about 'calculator' at runtime, not compile time.")
print(f"{'=' * 60}")

In [ ]:
# ============================================================
# Section 24 (continued) — MCP + Agent Integration
# ============================================================
# WHY: Show how MCP changes agent initialization
# WHY: Before MCP: tools hardcoded in the agent's TOOLS dict
# WHY: After MCP: agent discovers tools from MCP servers at startup
# WHY: Same agent logic, different tool source — that is the point of MCP

# WHY: requests for Ollama connection
import requests
import re
import json


def build_agent_from_mcp(mcp_client):
    """Build an agent's TOOLS dict from MCP-discovered tools.

    WHY: This replaces manual tool registration with dynamic discovery.
    WHY: Adding a new MCP server automatically adds its tools to the agent.
    WHY: The agent does not need to be rewritten — just reconnected.

    Args:
        mcp_client: An MCPClient connected to one or more servers.

    Returns:
        A TOOLS dict compatible with react_agent().
    """
    # WHY: Discover all available tools from all connected servers
    discovered = mcp_client.list_all_tools()
    mcp_tools = {}

    for tool in discovered:
        tool_name = tool["name"]
        # WHY: Build a wrapper function that calls the MCP client
        # WHY: The wrapper translates from our TOOLS interface to MCP's call_tool
        # WHY: Using default argument binding to avoid late-binding closure bug
        def make_wrapper(t_name):
            def wrapper(input_str):
                # WHY: MCP expects named arguments — wrap our string input
                result = mcp_client.call_tool(t_name, {"expression": input_str})
                # WHY: Extract text from MCP response format
                if "error" in result:
                    return f"Error: {result['error']}"
                return result["content"][0]["text"]
            return wrapper

        # WHY: Register in our standard TOOLS format for react_agent() compatibility
        mcp_tools[tool_name] = {
            "description": tool["description"],
            "fn": make_wrapper(tool_name),
        }

    return mcp_tools


def build_prompt_from_mcp(mcp_client):
    """Auto-generate the system prompt from MCP-discovered tools.

    WHY: The system prompt is built from whatever tools MCP provides.
    WHY: Add a new MCP server and the agent's prompt updates automatically.
    WHY: No manual prompt editing when tools change.
    """
    # WHY: Get tool metadata from MCP
    tools = mcp_client.list_all_tools()
    # WHY: Build tool descriptions for the system prompt
    tool_lines = "\n".join(
        f"  - {t['name']}(input): {t['description']}"
        for t in tools
    )
    # WHY: Same ReAct prompt structure — only the tool list is dynamic
    return (
        "You are a helpful assistant that uses tools step by step.\n\n"
        f"Available tools:\n{tool_lines}\n\n"
        "Format:\n"
        "Thought: <reasoning>\n"
        "Action: <tool_name>(<input>)\n\n"
        "When done:\n"
        "Thought: I have the answer.\n"
        "Final Answer: <answer>\n\n"
        "Rules: One tool per step. Wait for Observation. Do not guess."
    )


# ============================================================
# Create a multi-server MCP setup
# ============================================================
# WHY: Two servers — math and HR — each providing different tools
# WHY: The agent discovers ALL tools from ALL servers automatically

# WHY: Math server already created in previous cell — reuse it
# WHY: math_server has the calculator tool registered

# WHY: Create an HR server with the RAG lookup tool
hr_server = MCPServer("hr-tools")
# WHY: Register rag_lookup with MCP-compatible schema
hr_server.register_tool(
    name="rag_lookup",
    description="Search the Horizon Tech Employee Handbook for policy information",
    input_schema={
        "type": "object",
        "properties": {
            "expression": {"type": "string", "description": "Search query for employee handbook"}
        },
        "required": ["expression"],
    },
    # WHY: Delegate to the existing rag_lookup tool function from Part II
    fn=lambda expression: tool_rag_lookup(expression),
)

# WHY: Create a new client and connect to BOTH servers
mcp_agent_client = MCPClient()
mcp_agent_client.connect(math_server)
mcp_agent_client.connect(hr_server)

# WHY: Build agent tools from MCP discovery — dynamic, not hardcoded
mcp_agent_tools = build_agent_from_mcp(mcp_agent_client)

# WHY: Show what the agent discovered
print("\nMCP-discovered tools for agent:")
for name, info in mcp_agent_tools.items():
    print(f"  {name}: {info['description']}")

# WHY: Show the auto-generated prompt so the learner sees the connection
prompt = build_prompt_from_mcp(mcp_agent_client)
print(f"\nAuto-generated system prompt:\n{prompt}")

# ============================================================
# Run the agent with MCP-discovered tools
# ============================================================
# WHY: Same question as earlier demos, but tools come from MCP now
# WHY: The agent code (react_agent) is IDENTICAL — only the tool source changed
MCP_QUESTION = "What is the PTO policy at Horizon Tech?"

print(f"\n{'=' * 60}")
print(f"Running agent with MCP-discovered tools...")
print(f"Question: {MCP_QUESTION}")
print(f"{'=' * 60}")

try:
    # WHY: react_agent works with ANY tools dict — MCP-sourced or hardcoded
    answer = react_agent(MCP_QUESTION, mcp_agent_tools, max_steps=5, verbose=True)
    print(f"\nFINAL ANSWER: {answer}")
    # WHY: Print the architectural insight — what MCP enables
    print(f"\n{'=' * 60}")
    print("Same agent. Same code. Tools discovered via MCP at runtime.")
    print("Add a new MCP server \u2192 agent gains new capabilities automatically.")
    print(f"{'=' * 60}")
except requests.ConnectionError:
    # WHY: Ollama not running
    print("Error: Cannot connect to Ollama. Run: ollama serve")
except Exception as e:
    # WHY: Catch-all
    print(f"Error: {e}")

## Questions You Should Be Asking

You have now built multi-agent routers, seen framework equivalents, and implemented the MCP pattern. These questions should be forming:

1. **MCP vs. regular function calling — when does the abstraction pay off?** — MCP adds a discovery layer. For 2-3 hardcoded tools, it is overhead. For 10+ tools across multiple services, it is essential. Where is the crossover point for your system?

2. **What happens when an MCP server goes down mid-agent-run?** — The agent calls `call_tool()`, the server is unreachable, and the agent receives an error. How do you implement retry logic? Failover to a backup server? Circuit breaker patterns?

3. **What is the latency overhead of MCP vs. direct function calls?** — stdio transport adds microseconds. HTTP/SSE transport adds milliseconds to hundreds of milliseconds. For real-time agents, this matters. For batch processing, it does not.

4. **How do you secure MCP tools?** — Authentication (who can call?), authorization (which tools can they call?), rate limiting (how often?), and input validation (what arguments are allowed?). MCP servers need the same security as REST (Representational State Transfer) APIs.

5. **Can MCP servers call other MCP servers?** — Yes — nested tool composition. A "research" MCP server could call a "search" MCP server and a "summarize" MCP server internally. This creates tool hierarchies. How deep should nesting go before it becomes unmanageable?

6. **How do you version MCP tool schemas without breaking clients?** — Adding a new optional parameter is safe. Removing a required parameter breaks clients. This is the same API versioning problem as REST — semantic versioning, deprecation periods, backwards compatibility.

7. **stdio vs. HTTP transport — when to use which?** — stdio for local tools that run on the same machine (file system, local database). HTTP/SSE for remote tools (cloud APIs, shared services). Security considerations differ: stdio is sandboxed by the OS; HTTP requires TLS, auth tokens, and network security.

8. **How do you test MCP servers in isolation?** — Unit tests for each tool function. Integration tests for the server's `list_tools()` and `call_tool()` endpoints. Mock clients for end-to-end testing without a real LLM.

9. **What prevents a malicious MCP server from exfiltrating data?** — The client sends tool inputs to the server. If the server is compromised, it sees those inputs. Mitigation: run MCP servers in sandboxed environments, audit tool inputs, use allowlists for which servers can access which data.

10. **MCP vs. OpenAI function calling — what is the difference?** — OpenAI function calling is a proprietary format tied to OpenAI's API. MCP is an open protocol that works with any LLM. They solve the same problem (structured tool use) but MCP is framework-agnostic and supports dynamic discovery. OpenAI function calling requires tools to be defined at call time; MCP separates discovery from execution.

---

# Part V: Production & Integration

*The Restaurant Opens.*

---

You've built agents, advanced their reasoning, taught them to collaborate,
and standardized their tools. Now we open the restaurant to the public.
This section covers real-world architectures, evaluation, and clean code —
the difference between a test kitchen and a Michelin-starred restaurant.


## 25. Real-World Agent Architectures

Every production AI agent you use today follows one of the patterns we built
in this notebook. Here are seven products and the architectural patterns
behind them.

| Product | Pattern | What It Does | Notable Engineering |
|---|---|---|---|
| Atlassian Rovo | Hierarchical Router | Orchestrator dispatches to Jira / Confluence / Bitbucket subagents | Different system prompts for brainstorm vs. Q&A vs. deep reasoning modes |
| Databricks Genie | Pipeline | Intent → Schema → SQL → Validate | Each agent sees only the tables relevant to the query, reducing hallucination |
| Perplexity Deep Research | Planning + Reflection | Decompose → research 10-20 sources → synthesize → self-critique | Takes 30-60 s but produces substantially better results than single-pass |
| Claude Computer Use | Vision Agent | See screen → reason → click/type → verify result | Treats the entire computer as a tool via screenshots — no API required |
| GitHub Copilot Workspace | SDLC Pipeline | Spec → Plan → Implement → Review | Full software development lifecycle modeled as an agent pipeline |
| Devin | Persistent Environment | Long-running agent with shell, browser, and editor | Maintains state across hours of work; checkpoints and resumes |
| AutoGPT / BabyAGI | Autonomous | Self-directed goal decomposition and execution | Demonstrated possibilities *and* limitations of full autonomy |

> **Architect's Note — The Autonomy Pendulum.** The autonomous agent hype
> of 2023-2024 (AutoGPT, BabyAGI) generated enormous excitement but also
> revealed a critical lesson: unbounded autonomy is unreliable. By
> 2025-2026 the industry settled on **human-in-the-loop** as the dominant
> production pattern — agents propose, humans approve, agents execute.
> Claude Code, Cursor, and Copilot all implement this pattern. The chef
> cooks, but the customer confirms the order before the kitchen fires it.


## 26. Agent Evaluation — The Health Inspector and the Food Critic

Traditional software testing: `assert output == expected`.
Agent testing: assert *correct tools called* **AND** *answer contains
expected facts* **AND** *step count is reasonable*.

Agents are probabilistic — you test **properties**, not exact outputs.

| Dimension | What It Measures | Kitchen Analogy | Metrics |
|---|---|---|---|
| **Task Completion** | Did the agent answer correctly? | Food critic: "Does it taste right?" | Answer accuracy, fact correctness |
| **Efficiency** | How many steps / tokens did it take? | Kitchen manager: "How long did that take?" | Steps per answer, tokens per answer, wall-clock time |
| **Safety** | Did the agent do anything harmful? | Health inspector: "Any violations?" | No prompt injection, no tool misuse, no data leaks |

These three dimensions map directly to the evaluation harness we build next.


In [ ]:
# ============================================================
# Section 26 — Agent Evaluation Harness
# ============================================================
# WHY: Agents are probabilistic — exact-match testing doesn't work.
# WHY: We test three properties: completion, efficiency, safety.
# WHY: This mirrors how production teams evaluate agent systems.

from dataclasses import dataclass, field
from typing import List
import time

# ------------------------------------------------------------------
# Data classes for test cases and results
# ------------------------------------------------------------------

@dataclass
class TestCase:
    """One test case for the evaluation harness."""
    question: str                # WHY: The input the agent will receive
    expected_tools: List[str]    # WHY: Which tools should the agent call?
    expected_facts: List[str]    # WHY: What facts should appear in the answer?
    max_steps: int               # WHY: Efficiency bound — fail if exceeded

@dataclass
class TestResult:
    """Result of running one test case."""
    question: str
    tools_pass: bool             # WHY: Did the agent use the right tools?
    facts_pass: bool             # WHY: Did the answer contain expected facts?
    efficiency_pass: bool        # WHY: Did it stay within step bounds?
    actual_steps: int
    missing_tools: List[str]     # WHY: Diagnostics — what was missed
    missing_facts: List[str]     # WHY: Diagnostics — what was missed

# ------------------------------------------------------------------
# Define test cases
# ------------------------------------------------------------------
# WHY: Three test cases exercise different tool combinations:
#   1. calculator only
#   2. rag_lookup only
#   3. multi-tool (rag_lookup + calculator)

test_cases = [
    TestCase(
        question="What is 15% of 85000?",
        expected_tools=["calculator"],
        expected_facts=["12750"],
        max_steps=3
    ),
    TestCase(
        question="What is the PTO policy at Horizon Tech?",
        expected_tools=["rag_lookup"],
        expected_facts=["PTO", "days"],
        max_steps=4
    ),
    TestCase(
        question="How many PTO days does a new hire get, and what percentage of 260 working days is that?",
        expected_tools=["rag_lookup", "calculator"],
        expected_facts=["PTO"],
        max_steps=6
    ),
]

# ------------------------------------------------------------------
# Run evaluation
# ------------------------------------------------------------------
# WHY: We capture the agent's step log and final answer,
#      then check each dimension independently.

def evaluate_agent(test_cases, agent_fn, tools_dict, max_retries=1):
    """Run test cases through the agent and score each dimension."""
    results = []

    for tc in test_cases:
        print(f"\nTesting: {tc.question[:60]}...")

        try:
            # WHY: react_agent() returns (answer, steps) from Part II
            answer, steps = agent_fn(tc.question, tools_dict, max_steps=tc.max_steps + 2)

            # --- Dimension 1: Tool Completion ---
            # WHY: Check which tools the agent actually called
            tools_called = set()
            for step in steps:
                # WHY: Each step dict has an "action" key from the ReAct parser
                action = step.get("action", "")
                for tool_name in tools_dict:
                    if tool_name in action.lower():
                        tools_called.add(tool_name)

            missing_tools = [t for t in tc.expected_tools if t not in tools_called]
            tools_pass = len(missing_tools) == 0

            # --- Dimension 2: Fact Correctness ---
            # WHY: Substring check — flexible enough for probabilistic output
            answer_lower = answer.lower()
            missing_facts = [f for f in tc.expected_facts if f.lower() not in answer_lower]
            facts_pass = len(missing_facts) == 0

            # --- Dimension 3: Efficiency ---
            # WHY: Step count should be within the budget
            actual_steps = len(steps)
            efficiency_pass = actual_steps <= tc.max_steps

        except Exception as e:
            # WHY: If the agent crashes, all dimensions fail
            print(f"  Agent error: {e}")
            missing_tools = tc.expected_tools
            missing_facts = tc.expected_facts
            tools_pass = facts_pass = efficiency_pass = False
            actual_steps = -1

        results.append(TestResult(
            question=tc.question,
            tools_pass=tools_pass,
            facts_pass=facts_pass,
            efficiency_pass=efficiency_pass,
            actual_steps=actual_steps,
            missing_tools=missing_tools,
            missing_facts=missing_facts,
        ))

    return results


# ------------------------------------------------------------------
# Print report
# ------------------------------------------------------------------
# WHY: A clear report table makes pass/fail visible at a glance

def print_eval_report(results):
    """Print a formatted evaluation report."""
    # WHY: Header row for the report table
    print("\n" + "=" * 72)
    print("AGENT EVALUATION REPORT")
    print("=" * 72)
    print(f"{'Question':<42} {'Tools':>6} {'Facts':>6} {'Effic':>6} {'Steps':>6}")
    print("-" * 72)

    total_pass = 0
    total_tests = len(results)

    for r in results:
        # WHY: PASS/FAIL symbols for quick scanning
        t = "PASS" if r.tools_pass else "FAIL"
        f = "PASS" if r.facts_pass else "FAIL"
        e = "PASS" if r.efficiency_pass else "FAIL"
        all_pass = r.tools_pass and r.facts_pass and r.efficiency_pass
        if all_pass:
            total_pass += 1

        # WHY: Truncate long questions for table formatting
        q = r.question[:40] + ".." if len(r.question) > 42 else r.question
        print(f"{q:<42} {t:>6} {f:>6} {e:>6} {r.actual_steps:>6}")

        # WHY: Show diagnostics for failures
        if r.missing_tools:
            print(f"  -> Missing tools: {r.missing_tools}")
        if r.missing_facts:
            print(f"  -> Missing facts: {r.missing_facts}")

    print("-" * 72)
    print(f"Overall: {total_pass}/{total_tests} tests fully passed")
    print("=" * 72)

# ------------------------------------------------------------------
# Run the evaluation
# ------------------------------------------------------------------
# WHY: This ties together the test cases, agent function, and report

try:
    results = evaluate_agent(test_cases, react_agent, TOOLS)
    print_eval_report(results)
except Exception as e:
    # WHY: Graceful failure if Ollama is not running
    print(f"Evaluation error (is Ollama running with mistral?): {e}")


## "Why Does This Work?" — Agents as Software Systems

Agents are software systems, not magic. They need the same engineering
discipline as any production service: **testing, monitoring, logging,
and observability.**

**Property-based testing** works because agents are probabilistic. You
can't assert exact output, but you CAN assert:

- The **right tools** were called
- The answer **contains specific facts**
- The agent **terminated within bounds**
- No **dangerous tools** were invoked

This is the food critic's approach: they don't expect the dish to taste
*exactly* like last time, but they check that the flavor profile is right,
the ingredients are fresh, and it's served at the correct temperature.

The three evaluation dimensions — **completion, efficiency, safety** —
map directly to production monitoring:

| Evaluation Dimension | Production Metric |
|---|---|
| Task Completion | Success rate dashboard |
| Efficiency | Latency P50 / P99, token spend per query |
| Safety | Alert on unapproved tool calls, PII detection |


## 27. Clean Code — From Test Kitchen to Restaurant

We built everything from scratch to understand it. Now we package it for
production. A restaurant kitchen has stations, mise en place, and
standardized processes — not ingredients scattered across countertops.

| Before (scattered functions) | After (AgentPipeline class) |
|---|---|
| `TOOLS = {}` global dict | `self.tools: Dict[str, ToolSpec]` |
| `register_tool()` standalone function | `pipeline.register_tool()` method |
| `build_react_prompt()` standalone | `pipeline._build_system_prompt()` private |
| `react_agent()` standalone | `pipeline.query()` public method |
| Manual test loops | `pipeline.evaluate()` built-in |
| No configuration | `AgentConfig` dataclass |
| No structured output | `AgentStep` + `AgentResult` dataclasses |

The refactoring below wraps everything we built — tool registration, the
ReAct loop, response parsing, and evaluation — into a single
**AgentPipeline** class. This is the kitchen brigade pattern applied to
code: every responsibility has a home.


In [ ]:
# ============================================================
# Section 27 — AgentPipeline: Production-Ready Agent Class
# ============================================================
# WHY: Refactor scattered functions into a single cohesive class.
# WHY: This is the "kitchen brigade" applied to code — every
#      responsibility has a named home (method), not loose globals.

# WHY: dataclass reduces boilerplate for our config/result objects
from dataclasses import dataclass, field
# WHY: Type hints make the API self-documenting
from typing import Callable, Dict, List, Optional, Any
# WHY: Timestamps on each step enable debugging and audit trails
from datetime import datetime
# WHY: Regex for parsing LLM structured output (Thought/Action/Final Answer)
import re
# WHY: JSON for potential serialization of results
import json

# ------------------------------------------------------------------
# Supporting data classes
# ------------------------------------------------------------------
# WHY: Each data class represents one concept in the agent system.
# WHY: ToolSpec = what the agent can do
# WHY: AgentConfig = how the agent behaves
# WHY: AgentStep = one iteration of reasoning
# WHY: AgentResult = the complete output package

@dataclass
class ToolSpec:
    """Metadata + callable for one tool."""
    name: str              # WHY: Unique identifier for the tool
    description: str       # WHY: Injected into the system prompt so the LLM knows what's available
    fn: Callable           # WHY: The actual function to execute

@dataclass
class AgentConfig:
    """Configuration for the agent pipeline."""
    # WHY: Defaults let you create AgentConfig() with zero arguments
    model: str = "mistral"          # WHY: Default to local Mistral via Ollama
    max_steps: int = 8              # WHY: Safety bound — prevent infinite loops
    temperature: float = 0.1        # WHY: Low temperature for deterministic tool use

@dataclass
class AgentStep:
    """One iteration of the ReAct loop."""
    step_number: int
    thought: str                    # WHY: The agent's reasoning (Thought:)
    action: str                     # WHY: The tool call or Final Answer
    observation: str                # WHY: The tool's return value
    timestamp: str = ""             # WHY: For logging and debugging

    def __post_init__(self):
        # WHY: Auto-stamp each step for audit trails
        # WHY: In production, this enables latency analysis per step
        if not self.timestamp:
            self.timestamp = datetime.now().isoformat()

@dataclass
class AgentResult:
    """Complete result from a pipeline query."""
    question: str
    answer: str                     # WHY: The final answer returned to the user
    steps: List[AgentStep]          # WHY: Full reasoning trace for debugging
    total_tokens_estimate: int = 0  # WHY: Rough cost tracking
    success: bool = True            # WHY: False if agent hit max steps or errored

# ------------------------------------------------------------------
# AgentPipeline — the main class
# ------------------------------------------------------------------
# WHY: A single class that owns the full agent lifecycle:
#      register tools → build prompt → run loop → return result → evaluate
# WHY: This replaces the scattered global functions from earlier sections

class AgentPipeline:
    """
    Production-ready ReAct agent pipeline.

    Usage:
        pipeline = AgentPipeline(AgentConfig())
        pipeline.register_tool("calculator", "Evaluate math", calc_fn)
        result = pipeline.query("What is 2 + 2?")
        print(result.answer)
    """

    def __init__(self, config: Optional[AgentConfig] = None):
        # WHY: Default config if none provided — sensible defaults
        self.config = config or AgentConfig()
        # WHY: Tool registry maps name → ToolSpec (like a restaurant menu)
        # WHY: Dict allows O(1) lookup when dispatching tool calls
        self.tools: Dict[str, ToolSpec] = {}

    def register_tool(self, name: str, description: str, fn: Callable):
        """Register a tool for the agent to use."""
        # WHY: Centralized registration — tools are data, not hardcoded
        # WHY: Same pattern as MCP — tools are discoverable, not embedded
        self.tools[name] = ToolSpec(name=name, description=description, fn=fn)

    def _build_system_prompt(self) -> str:
        """Auto-generate the system prompt from registered tools."""
        # WHY: The system prompt is generated, not handwritten.
        # WHY: Adding a tool automatically updates the prompt — zero manual sync.
        # WHY: This is the key insight: tool descriptions ARE the prompt.
        tool_lines = []
        for name, spec in self.tools.items():
            tool_lines.append(f"  - {name}: {spec.description}")
        tools_block = "\n".join(tool_lines)

        # WHY: The ReAct format is explicit so the LLM follows it reliably
        return f"""You are a helpful assistant. You can use these tools:
{tools_block}

For each step, respond in EXACTLY this format:
Thought: <your reasoning>
Action: <tool_name>(<input>)

When you have the final answer:
Thought: <your reasoning>
Final Answer: <your answer>

Rules:
- Always think before acting.
- Use tools when you need external information or computation.
- Never guess when a tool can give you the exact answer."""

    def _parse_response(self, text: str) -> dict:
        """Extract Thought, Action, and Final Answer from LLM output."""
        # WHY: Structured parsing turns free text into actionable data
        # WHY: Regex is more robust than string.split() for LLM output
        # WHY: Empty defaults ensure downstream code never hits KeyError
        result = {"thought": "", "action": "", "final_answer": ""}

        # WHY: Extract the Thought line
        thought_match = re.search(r"Thought:\s*(.+?)(?=\n(?:Action|Final))", text, re.DOTALL)
        if thought_match:
            result["thought"] = thought_match.group(1).strip()

        # WHY: Check for Final Answer first — it terminates the loop
        final_match = re.search(r"Final Answer:\s*(.+)", text, re.DOTALL)
        if final_match:
            result["final_answer"] = final_match.group(1).strip()
            return result

        # WHY: Extract the Action (tool call)
        action_match = re.search(r"Action:\s*(.+?)(?=\n|$)", text)
        if action_match:
            result["action"] = action_match.group(1).strip()

        return result

    def _dispatch_tool(self, action_str: str) -> str:
        """Parse tool name and input from action string, execute the tool."""
        # WHY: Robust parsing handles malformed LLM output gracefully
        try:
            # WHY: Expected format is tool_name(input)
            match = re.match(r"(\w+)\((.*)\)", action_str)
            if not match:
                return f"Error: Could not parse action: {action_str}"

            tool_name = match.group(1).strip()
            tool_input = match.group(2).strip().strip("'\"")

            # WHY: Check if tool exists before calling
            if tool_name not in self.tools:
                return f"Error: Unknown tool '{tool_name}'. Available: {list(self.tools.keys())}"

            # WHY: Execute with error handling — tools can fail
            result = self.tools[tool_name].fn(tool_input)
            return str(result)

        except Exception as e:
            # WHY: Never let a tool crash the entire pipeline
            return f"Tool error: {e}"

    def query(self, question: str) -> AgentResult:
        """Run the full ReAct loop and return a structured result."""
        # WHY: This is the heart of the agent — the ReAct loop
        # WHY: Build the prompt from current tool registry
        system_prompt = self._build_system_prompt()
        # WHY: Steps list becomes the full audit trail
        steps = []
        # WHY: Conversation history accumulates for multi-step reasoning
        # WHY: Each iteration sees ALL prior steps — the agent has memory
        conversation = f"Question: {question}"
        # WHY: Token tracking enables cost analysis in production
        token_estimate = 0

        try:
            for step_num in range(1, self.config.max_steps + 1):
                # WHY: Send accumulated context to the LLM
                # WHY: Prompt grows each iteration — watch for context window limits
                full_prompt = f"{system_prompt}\n\n{conversation}"
                response = ollama_generate(
                    full_prompt,
                    model=self.config.model,
                    temperature=self.config.temperature
                )

                # WHY: Rough token estimate for cost tracking
                token_estimate += len(full_prompt.split()) + len(response.split())

                # WHY: Parse the LLM's structured response
                parsed = self._parse_response(response)

                # WHY: Check for final answer — loop termination
                if parsed["final_answer"]:
                    steps.append(AgentStep(
                        step_number=step_num,
                        thought=parsed["thought"],
                        action="Final Answer",
                        observation=parsed["final_answer"]
                    ))
                    return AgentResult(
                        question=question,
                        answer=parsed["final_answer"],
                        steps=steps,
                        total_tokens_estimate=token_estimate,
                        success=True
                    )

                # WHY: No final answer yet — dispatch the tool
                # WHY: The agent chose to act; execute and observe
                if parsed["action"]:
                    observation = self._dispatch_tool(parsed["action"])
                else:
                    # WHY: Nudge the LLM if it didn't produce a valid action
                    observation = "No action specified. Please use a tool or provide a Final Answer."

                # WHY: Record this step for the audit trail
                steps.append(AgentStep(
                    step_number=step_num,
                    thought=parsed["thought"],
                    action=parsed["action"],
                    observation=observation
                ))

                # WHY: Append to conversation so next iteration has full history
                # WHY: This is how the agent "remembers" prior steps
                conversation += f"""
Thought: {parsed['thought']}
Action: {parsed['action']}
Observation: {observation}
"""

            # WHY: If we exhaust max_steps, return what we have
            return AgentResult(
                question=question,
                answer="Agent reached maximum steps without a final answer.",
                steps=steps,
                total_tokens_estimate=token_estimate,
                success=False
            )

        except Exception as e:
            # WHY: Graceful failure — return partial result with error info
            return AgentResult(
                question=question,
                answer=f"Agent error: {e}",
                steps=steps,
                total_tokens_estimate=token_estimate,
                success=False
            )

    def evaluate(self, test_cases: list) -> list:
        """Run test cases and return evaluation results."""
        # WHY: Built-in evaluation — every pipeline should be testable
        # WHY: Same three dimensions: completion, efficiency, safety
        results = []
        for tc in test_cases:
            # WHY: Run the agent on this test case's question
            result = self.query(tc.question)

            # WHY: Check tool usage from step logs (completion dimension)
            # WHY: We scan action strings for tool names
            tools_used = set()
            for step in result.steps:
                for tool_name in self.tools:
                    if tool_name in step.action.lower():
                        tools_used.add(tool_name)

            # WHY: Compare actual vs expected tools
            missing_tools = [t for t in tc.expected_tools if t not in tools_used]
            tools_pass = len(missing_tools) == 0

            # WHY: Fact checking via substring match (flexible for LLM output)
            # WHY: Property-based — we check presence, not exact wording
            answer_lower = result.answer.lower()
            missing_facts = [f for f in tc.expected_facts if f.lower() not in answer_lower]
            facts_pass = len(missing_facts) == 0

            # WHY: Efficiency check — did we stay within step budget?
            efficiency_pass = len(result.steps) <= tc.max_steps

            # WHY: Structured dict makes report generation straightforward
            results.append({
                "question": tc.question,
                "tools_pass": tools_pass,
                "facts_pass": facts_pass,
                "efficiency_pass": efficiency_pass,
                "steps": len(result.steps),
                "answer": result.answer[:80],
                "missing_tools": missing_tools,
                "missing_facts": missing_facts,
            })
        return results


# ------------------------------------------------------------------
# Demo: Create pipeline, register tools, run queries
# ------------------------------------------------------------------
# WHY: Demonstrate the clean API matches the scattered functions' behavior
# WHY: Same questions as before — proof that refactoring preserved behavior

try:
    # WHY: Create pipeline with default config (mistral, max 8 steps)
    # WHY: AgentConfig makes all settings explicit and changeable
    pipeline = AgentPipeline(AgentConfig(model="mistral", max_steps=8))

    # WHY: Register the same tools from earlier sections
    # WHY: The TOOLS dict from Part II is reused — tools are portable
    # WHY: register_tool wraps raw functions into ToolSpec with metadata
    pipeline.register_tool(
        "calculator",
        "Evaluate a math expression. Input: math string like '2 + 2'",
        TOOLS["calculator"]
    )
    # WHY: Reusing TOOLS dict proves tools are portable between architectures
    pipeline.register_tool(
        "rag_lookup",
        "Look up company policy or employee handbook info. Input: search query",
        TOOLS["rag_lookup"]
    )

    # WHY: Run the same questions to show identical behavior, cleaner API
    # WHY: The difference is the return type: AgentResult (structured) vs raw tuple
    demo_questions = [
        "What is 15% of 85000?",
        "What is the PTO policy at Horizon Tech?",
    ]

    for q in demo_questions:
        print(f"\nQuestion: {q}")
        # WHY: pipeline.query() replaces react_agent() — same loop, better packaging
        result = pipeline.query(q)
        print(f"Answer: {result.answer}")
        print(f"Steps: {len(result.steps)} | Success: {result.success}")
        # WHY: Show the structured step trace for debugging
        # WHY: Each step has a timestamp — useful for latency analysis
        for step in result.steps:
            print(f"  Step {step.step_number}: {step.action} -> {step.observation[:60]}")

    print("\nAgentPipeline demo complete.")

except Exception as e:
    # WHY: Graceful failure if Ollama is not running
    print(f"Pipeline demo error (is Ollama running with mistral?): {e}")
    print("The AgentPipeline class is defined and ready — start Ollama to run it.")


## The Complete Agent Pipeline

```
┌──────────────────────────────────────────────────────────┐
│                    AGENT PIPELINE                         │
│                                                           │
│  ┌──────────┐    ┌───────────────────┐    ┌──────────┐  │
│  │  User     │───▶│  AgentPipeline    │───▶│  Result   │  │
│  │  Query    │    │                   │    │  + Steps  │  │
│  └──────────┘    │  ┌─────────────┐  │    └──────────┘  │
│                   │  │ Tool        │  │                   │
│                   │  │ Registry    │  │                   │
│                   │  │ ┌─────────┐ │  │                   │
│                   │  │ │calc     │ │  │                   │
│                   │  │ │rag      │ │  │                   │
│                   │  │ │datetime │ │  │                   │
│                   │  │ │analyze  │ │  │                   │
│                   │  │ └─────────┘ │  │                   │
│                   │  └─────────────┘  │                   │
│                   │                   │                   │
│                   │  ┌─────────────┐  │                   │
│                   │  │ ReAct Loop  │  │                   │
│                   │  │ Think→Act   │  │                   │
│                   │  │ →Observe    │  │                   │
│                   │  └─────────────┘  │                   │
│                   └───────────────────┘                   │
│                                                           │
│  ┌───────────────────────────────────────────────────┐   │
│  │ MCP Servers (optional)                             │   │
│  │  math_server │ hr_server │ datetime_server         │   │
│  └───────────────────────────────────────────────────┘   │
│                                                           │
│  ┌───────────────────────────────────────────────────┐   │
│  │ Evaluation Harness                                 │   │
│  │  TestCase[] → Run → Report (completion/efficiency) │   │
│  └───────────────────────────────────────────────────┘   │
└──────────────────────────────────────────────────────────┘
```

Every component in this diagram corresponds to code you wrote in this
notebook. The pipeline is not a framework — it's *your* code, organized
for production.


## Self-Check: Can You Explain These?

1. What is the difference between ReAct and Chain of Thought?
2. Name the three essential components every agent needs.
3. What happens when you remove "Thought:" from a ReAct prompt?
4. How does a tool registry auto-generate the system prompt?
5. When would you use Tree of Thought instead of ReAct?
6. What does a Planning Loop do that ReAct doesn't?
7. Why does specialization (multi-agent) reduce errors?
8. What problem does MCP solve?
9. Name the three dimensions of agent evaluation.
10. When should you use LangGraph vs. pure Python?
11. Why is human-in-the-loop the dominant production pattern?
12. Draw the ReAct loop from memory (Thought → Action → Observation → repeat).

If you can answer all 12, you're ready for production agent development.


## What Comes Next

| Topic | What You'll Learn | Where |
|---|---|---|
| **Deployment** | Serve agents as APIs, handle concurrent users, scale | Production Engineering module |
| **Agent-to-Agent Communication** | A2A protocol, agent discovery, delegation across services | Advanced Agents |
| **MCP Tool Suite** | Build a library of MCP servers (DB, calendar, email, Slack) | MCP Deep Dive |
| **Evaluation Deep Dive** | Benchmarks (SWE-bench, HumanEval, GAIA), automated evals | Evaluation module |
| **Long-Horizon Agents** | Agents that run for hours, checkpoint and resume | Advanced Agents |
| **Tool Governance** | Approval workflows, audit trails, rate limiting | Production Engineering |

Each topic builds on the foundation from this notebook. The ReAct loop,
tool registry, multi-agent coordination, and evaluation harness you built
here are the primitives that everything above uses.


## Project Ideas

### Personal Research Agent ★

Build an agent that takes a research question, searches multiple sources,
and produces a summary with citations.

**Tools:** `web_search`, `summarize`, `cite`
**Pattern:** Planning Loop — decompose the question into sub-queries,
research each, synthesize results.

---

### Expense Report Agent ★★

Build an agent that reads expense receipts (images), extracts amounts and
categories, validates against company policy, and generates a report.

**Combines:** Multimodal (image reading) + Agents (validation logic) +
RAG (policy lookup)
**Pattern:** Pipeline — extract → classify → validate → report

---

### Code Review Agent ★★★

Build an agent that reads a git diff, identifies potential issues (bugs,
style violations, security concerns), and generates review comments.

**Tools:** `read_file`, `search_codebase`, `check_style`, `security_scan`
**Pattern:** Multi-agent — specialist reviewers (style, security, logic)
coordinated by an orchestrator.

---

### Portfolio Project: IT Helpdesk Agent ★★★

Build a multi-agent system that routes IT support tickets (password resets,
VPN issues, software requests) to specialized agents, each with their own
tools and knowledge base. Add evaluation and MCP for tool discovery.

**Pattern:** Hierarchical Router + MCP tool registry
**This project demonstrates:** tool use, multi-agent coordination,
evaluation, and production patterns — ideal for a portfolio.


In [ ]:
# ============================================================
# Cleanup — Remove temporary data
# ============================================================
# WHY: ChromaDB creates local persistence files during the notebook
# WHY: Clean up to avoid accumulating data across repeated runs
# WHY: A clean kitchen is a professional kitchen

import shutil
import os

# WHY: These paths may have been created by ChromaDB or agent tools
cleanup_paths = [
    "./chroma_data",
    "./.chroma",
    "./agent_chroma_db",
]

cleaned = 0
for path in cleanup_paths:
    # WHY: Only remove paths that actually exist
    if os.path.exists(path):
        shutil.rmtree(path)
        cleaned += 1
        print(f"  Removed: {path}")

if cleaned == 0:
    print("Nothing to clean up.")
else:
    print(f"\nCleaned {cleaned} temporary directories.")

print("Notebook complete. The kitchen is clean.")


## Ten Things We Didn't Cover

1. **Memory Persistence** — Agents that remember across sessions (vector DB,
   graph DB, key-value stores). Today's agent forgets everything when the
   notebook restarts.

2. **Agent-to-Agent Communication** — The A2A protocol for agents
   discovering and delegating to each other across organizational boundaries.

3. **Self-Improving Agents** — Agents that learn from feedback and update
   their own prompts. Meta-learning applied to tool use.

4. **Vision Agents** — Agents that see screens and interact with GUIs
   (Claude Computer Use). The screen *is* the tool.

5. **Code Sandboxing** — Running agent-generated code safely (Docker, E2B,
   Modal). Essential when agents write and execute their own code.

6. **Streaming Output** — Showing agent reasoning in real-time
   (token-by-token streaming). Critical for user experience in production.

7. **Cost Optimization** — Caching, prompt compression, model routing (fast
   model for routing, smart model for reasoning). A production agent system
   can burn through API budgets without this.

8. **Compliance & Audit** — Logging every tool call for regulatory
   requirements (SOC 2, HIPAA). Every `AgentStep` becomes an audit record.

9. **Federated Agents** — Agents distributed across organizations with
   privacy boundaries. Company A's agent delegates to Company B's agent
   without sharing raw data.

10. **Benchmarks** — SWE-bench, HumanEval, GAIA, AgentBench — standardized
    ways to measure agent capability across tasks.

Each of these is a notebook in itself. The foundation you built today makes
all of them accessible.


## Glossary

| Term | Definition |
|---|---|
| **Agent** | An AI system that reasons, plans, uses tools, and acts in a loop until a goal is achieved |
| **ReAct** | Reason + Act — interleaving thinking with tool use (Yao et al., 2022) |
| **Tool** | A function with metadata that an agent can invoke (calculator, search, API call) |
| **Orchestrator** | An agent that decomposes goals and delegates to specialist agents |
| **MCP** | Model Context Protocol — standardized interface for AI tool discovery and execution |
| **Tree of Thought** | Generate multiple candidate solutions, evaluate each, select the best |
| **Planning Loop** | Decompose → plan subtasks → execute each → synthesize results |
| **Reflection** | Execute → evaluate own output → critique → improve |
| **LangGraph** | A framework for building stateful agent workflows as graphs |
| **Tool Registry** | A dictionary mapping tool names to their descriptions and functions |
| **Router** | An agent that classifies queries and dispatches to specialists |
| **Kitchen Brigade** | Multi-agent pattern where specialists coordinate under an orchestrator |
| **Human-in-the-Loop** | Production pattern where agents propose actions, humans approve before execution |


## Final Architect's Note

> *The kitchen is open.*
>
> You started with a chef who can read an order and pick up a knife. You
> taught that chef to reason before acting (ReAct), consider multiple
> approaches before committing (Tree of Thought), break complex meals into
> courses (Planning Loops), and coordinate with a brigade of specialists
> (Multi-Agent). You standardized the kitchen layout so any chef can work
> in any kitchen (MCP). And you built an evaluation system so the health
> inspector and food critic can verify quality.
>
> The progression: **RAG** taught your AI to find information.
> **Multimodal** taught it to perceive. **Agents** taught it to act. The
> combination — an AI that can see, hear, retrieve knowledge, reason,
> plan, and execute — is the foundation of every production AI system
> shipping today.
>
> Next: **Deployment** — the restaurant that serves a thousand tables
> without the kitchen catching fire.


## Taking Agents to Production

The agent we built works in a notebook. Here's what changes when real users depend on it.

### Deployment Checklist

| Concern | Notebook | Production |
|:--------|:---------|:-----------|
| **Where it runs** | Your laptop | Docker container on cloud (AWS ECS/GCP Cloud Run) |
| **How users reach it** | `agent.run()` in a cell | REST API (FastAPI) or WebSocket for streaming |
| **Tool failures** | Crash the cell | Retry with backoff, fallback response, alert on-call |
| **Cost** | Free (local model) | Per-token billing → budget alerts, model routing |
| **Latency** | "It's fine" | P95 < 3 seconds → cache, streaming (SSE), smaller models for simple queries |
| **Safety** | Trust yourself | Input guardrails, output filtering, PII detection, audit logging |
| **Observability** | `print()` statements | Structured logging, trace IDs per request, LLM call dashboards |

### The Troubleshooting Pattern (Sunil's Methodology)

When an agent fails in production:

1. **Identify the symptom:** "Agent returned wrong tool" / "Agent hallucinated" / "Agent timed out"
2. **Trace the request:** Follow the trace ID through logs → which tool was called? What was the LLM prompt? What did the LLM return?
3. **Narrow the area:** Is it routing (wrong tool chosen)? Execution (tool failed)? Generation (LLM output bad)?
4. **Add targeted logging:** In the narrowed area, add detailed print/log statements
5. **Reproduce with the exact input:** Use the logged prompt to reproduce locally
6. **Fix and add a regression test:** The exact input that broke becomes a test case

> **This is the same print-statement drill-down methodology used in all debugging.** Start broad (which component?), narrow down (which step?), then drill into the exact line. The issue surfaces naturally.

### Monitoring in Production

```
Key metrics to track:
- Tool selection accuracy (% of queries routed to correct tool)
- End-to-end latency (P50, P95, P99)
- Error rate by tool (which tools fail most?)
- Token usage per query (cost monitoring)
- User satisfaction (thumbs up/down if UI supports it)
```
